# YOLO11m: resume the existing Phase 1 experiment

Use this only in a fresh free Kaggle GPU session after confirming Colab is stopped. Do not run setup cells during active training. Attach the private labels-v3 train/validation dataset and epoch-30 recovery input. Enable internet and notebook access to the authorized Secret `MLOPS_EL_KAGGLE_API_TOKEN`; never put its value in notebook code.

Run ID: `85714466948d4e658ebb610a475dcf57`. YOLO11m, 960px, batch4, max100epochs, patience20, 7000 training / 1500 validation images; test excluded. The last cell restores the latest private remote checkpoint rather than starting a new run. Backup failure stops continuation. Do not run first-time initialization again.

In [ ]:
import sys, subprocess, pathlib, importlib.metadata as md
import torch
assert torch.cuda.is_available(), 'Select a free GPU; never use CPU or paid fallback.'
assert md.version('torch') == '2.11.0+cu130'
assert md.version('torchvision') == '0.26.0+cu130'
p=pathlib.Path('/kaggle/working/mlops-dependency-constraints.txt')
p.write_text('torch==2.11.0+cu130\ntorchvision==0.26.0+cu130\n')
subprocess.check_call([sys.executable,'-m','pip','install','-q','-c',str(p),'ultralytics==8.4.201','mlflow==3.17.2','PyYAML==6.0.3','kaggle==2.0.2'])
print({n:md.version(n) for n in ['torch','torchvision','ultralytics','mlflow','kaggle']})


In [ ]:
import base64, pathlib, sys
code_dir=pathlib.Path('/kaggle/working/mlops_code'); code_dir.mkdir(exist_ok=True)
embedded={"runner.py": "IiIiRnJlZS1DVURBIGJlbmNobWFyay90cmFpbmluZyBoZWxwZXJzLiBGdWxsIHRyYWluaW5nIGlzIGV4cGxpY2l0bHkgZ2F0ZWQuIiIiCmltcG9ydCBhcmdwYXJzZQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgsIFB1cmVQb3NpeFBhdGgKaW1wb3J0IHBsYXRmb3JtCmltcG9ydCBzaHV0aWwKaW1wb3J0IHNxbGl0ZTMKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRhcmZpbGUKaW1wb3J0IHRpbWUKCgpkZWYgc2hhKHBhdGgpOgogICAgaD1oYXNobGliLnNoYTI1NigpCiAgICB3aXRoIFBhdGgocGF0aCkub3BlbigncmInKSBhcyBmOgogICAgICAgIGZvciBibG9jayBpbiBpdGVyKGxhbWJkYTpmLnJlYWQoMTAyNCoxMDI0KSxiJycpOmgudXBkYXRlKGJsb2NrKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgc2FmZV9uYW1lKG5hbWUpOgogICAgcD1QdXJlUG9zaXhQYXRoKG5hbWUpCiAgICBpZiBwLmlzX2Fic29sdXRlKCkgb3IgJy4uJyBpbiBwLnBhcnRzIG9yIG5vdCBwLnBhcnRzIG9yICdcXCcgaW4gbmFtZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVbnNhZmUgYXJjaGl2ZSBwYXRoJykKICAgIHJldHVybiBwCgoKZGVmIHVucGFjayhhcmNoaXZlLGRlc3RpbmF0aW9uLGV4cGVjdGVkX3NoYSk6CiAgICBhcmNoaXZlLGRlc3RpbmF0aW9uPVBhdGgoYXJjaGl2ZSksUGF0aChkZXN0aW5hdGlvbikKICAgIGlmIHNoYShhcmNoaXZlKSE9ZXhwZWN0ZWRfc2hhOnJhaXNlIFZhbHVlRXJyb3IoJ0FyY2hpdmUgY2hlY2tzdW0gbWlzbWF0Y2gnKQogICAgaWYgZGVzdGluYXRpb24uZXhpc3RzKCk6cmFpc2UgVmFsdWVFcnJvcignRXh0cmFjdGlvbiBkZXN0aW5hdGlvbiBleGlzdHM7IHVzZSBhIGZyZXNoIGRpcmVjdG9yeScpCiAgICB3aXRoIHRhcmZpbGUub3BlbihhcmNoaXZlLCdyOicpIGFzIHRhcjoKICAgICAgICBtZW1iZXJzPXRhci5nZXRtZW1iZXJzKCk7bmFtZXM9W20ubmFtZSBmb3IgbSBpbiBtZW1iZXJzXQogICAgICAgIGlmIGxlbihuYW1lcykhPWxlbihzZXQobmFtZXMpKTpyYWlzZSBWYWx1ZUVycm9yKCdEdXBsaWNhdGUgYXJjaGl2ZSBlbnRyaWVzJykKICAgICAgICBmb3IgbSBpbiBtZW1iZXJzOgogICAgICAgICAgICBzYWZlX25hbWUobS5uYW1lKQogICAgICAgICAgICBpZiBub3QgbS5pc2ZpbGUoKTpyYWlzZSBWYWx1ZUVycm9yKCdPbmx5IHJlZ3VsYXIgZmlsZXMgYXJlIGFjY2VwdGVkOyBsaW5rcyBhcmUgcmVqZWN0ZWQnKQogICAgICAgIG1hbmlmZXN0PWpzb24ubG9hZCh0YXIuZXh0cmFjdGZpbGUoJ21hbmlmZXN0Lmpzb24nKSkKICAgICAgICBleHBlY3RlZD17clsncGF0aCddOnIgZm9yIHIgaW4gbWFuaWZlc3RbJ2ZpbGVzJ119CiAgICAgICAgaWYgbGVuKGV4cGVjdGVkKSE9bGVuKG1hbmlmZXN0WydmaWxlcyddKTpyYWlzZSBWYWx1ZUVycm9yKCdEdXBsaWNhdGUgbWFuaWZlc3QgcGF0aHMnKQogICAgICAgIGlmIHNldChuYW1lcykhPXNldChleHBlY3RlZCl8eydtYW5pZmVzdC5qc29uJ306cmFpc2UgVmFsdWVFcnJvcignQXJjaGl2ZSBtZW1iZXJzIGRpZmZlciBmcm9tIG1hbmlmZXN0JykKICAgICAgICBkZXN0aW5hdGlvbi5ta2RpcihwYXJlbnRzPVRydWUpCiAgICAgICAgZm9yIG0gaW4gbWVtYmVyczoKICAgICAgICAgICAgaWYgbS5uYW1lPT0nbWFuaWZlc3QuanNvbic6Y29udGludWUKICAgICAgICAgICAgcm93PWV4cGVjdGVkW20ubmFtZV0KICAgICAgICAgICAgaWYgcm93WydieXRlcyddIT1tLnNpemU6cmFpc2UgVmFsdWVFcnJvcignQXJjaGl2ZSBzaXplIG1pc21hdGNoJykKICAgICAgICAgICAgdGFyZ2V0PWRlc3RpbmF0aW9uLmpvaW5wYXRoKCpzYWZlX25hbWUobS5uYW1lKS5wYXJ0cyk7dGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgaD1oYXNobGliLnNoYTI1NigpCiAgICAgICAgICAgIHdpdGggdGFyLmV4dHJhY3RmaWxlKG0pIGFzIHNyYyx0YXJnZXQub3Blbignd2InKSBhcyBkc3Q6CiAgICAgICAgICAgICAgICBmb3IgYmxvY2sgaW4gaXRlcihsYW1iZGE6c3JjLnJlYWQoMTAyNCoxMDI0KSxiJycpOgogICAgICAgICAgICAgICAgICAgIGgudXBkYXRlKGJsb2NrKTtkc3Qud3JpdGUoYmxvY2spCiAgICAgICAgICAgIGlmIGguaGV4ZGlnZXN0KCkhPXJvd1snc2hhMjU2J106cmFpc2UgVmFsdWVFcnJvcihmJ0ZpbGUgY2hlY2tzdW0gbWlzbWF0Y2g6IHttLm5hbWV9JykKICAgICAgICAoZGVzdGluYXRpb24vJ21hbmlmZXN0Lmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMobWFuaWZlc3QsaW5kZW50PTIpKQogICAgdmVyaWZ5X2J1bmRsZShkZXN0aW5hdGlvbikKICAgIHJldHVybiBtYW5pZmVzdAoKCmRlZiB2ZXJpZnlfYnVuZGxlKHJvb3QpOgogICAgcm9vdD1QYXRoKHJvb3QpO21hbmlmZXN0PWpzb24ubG9hZHMoKHJvb3QvJ21hbmlmZXN0Lmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGZvciByb3cgaW4gbWFuaWZlc3RbJ2ZpbGVzJ106CiAgICAgICAgcGF0aD1yb290LmpvaW5wYXRoKCpzYWZlX25hbWUocm93WydwYXRoJ10pLnBhcnRzKQogICAgICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKSBvciBwYXRoLnN0YXQoKS5zdF9zaXplIT1yb3dbJ2J5dGVzJ10gb3Igc2hhKHBhdGgpIT1yb3dbJ3NoYTI1NiddOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnQnVuZGxlIHZlcmlmaWNhdGlvbiBmYWlsZWQ6IHtyb3dbInBhdGgiXX0nKQogICAgaWYgYW55KCcvdGVzdC8nIGluIHJbJ3BhdGgnXSBmb3IgciBpbiBtYW5pZmVzdFsnZmlsZXMnXSk6cmFpc2UgVmFsdWVFcnJvcignVGVzdCBkYXRhIG11c3Qgbm90IGJlIGV4cG9ydGVkJykKICAgIGV4cGVjdGVkPXsnYmVuY2htYXJrJzooMjU2LDMyKSwndHJhaW52YWwnOig3MDAwLDE1MDApfVttYW5pZmVzdFsnc2NvcGUnXV0KICAgIGlmIChtYW5pZmVzdFsndHJhaW5faW1hZ2VzJ10sbWFuaWZlc3RbJ3ZhbF9pbWFnZXMnXSkhPWV4cGVjdGVkOnJhaXNlIFZhbHVlRXJyb3IoJ1VuZXhwZWN0ZWQgZGF0YXNldCBzaXplJykKICAgIGNvdW50cz1Db3VudGVyKCcvJy5qb2luKHJbJ3BhdGgnXS5zcGxpdCgnLycpWzozXSkgZm9yIHIgaW4gbWFuaWZlc3RbJ2ZpbGVzJ10pCiAgICBmb3Igc3BsaXQsY291bnQgaW4gemlwKCgndHJhaW4nLCd2YWwnKSxleHBlY3RlZCk6CiAgICAgICAgaWYgY291bnRzW2YnZGF0YXNldC9pbWFnZXMve3NwbGl0fSddIT1jb3VudCBvciBjb3VudHNbZidkYXRhc2V0L2xhYmVscy97c3BsaXR9J10hPWNvdW50OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdBY3R1YWwgaW1hZ2UvbGFiZWwgY291bnRzIGRpZmZlciBmcm9tIGRlY2xhcmVkIHNjb3BlJykKICAgIHJldHVybiBtYW5pZmVzdAoKCmRlZiBhdG9taWNfY29weShzb3VyY2UsZGVzdGluYXRpb24pOgogICAgZGVzdGluYXRpb249UGF0aChkZXN0aW5hdGlvbik7ZGVzdGluYXRpb24ucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgdGVtcG9yYXJ5PWRlc3RpbmF0aW9uLndpdGhfbmFtZShkZXN0aW5hdGlvbi5uYW1lKycucGFydCcpCiAgICBzaHV0aWwuY29weWZpbGUoc291cmNlLHRlbXBvcmFyeSkKICAgIGlmIHNoYSh0ZW1wb3JhcnkpIT1zaGEoc291cmNlKTpyYWlzZSBJT0Vycm9yKCdQZXJzaXN0ZW50IGNvcHkgdmVyaWZpY2F0aW9uIGZhaWxlZCcpCiAgICB0ZW1wb3JhcnkucmVwbGFjZShkZXN0aW5hdGlvbikKCgpkZWYgc2F2ZV9jaGVja3BvaW50X2dlbmVyYXRpb24odHJhaW5lcixwZXJzaXN0ZW50LG1ldGFkYXRhKToKICAgICIiIktlZXAgdHdvIGNvbXBsZXRlIGNoZWNrcG9pbnQgZ2VuZXJhdGlvbnM7IHB1Ymxpc2ggcG9pbnRlciBvbmx5IGFmdGVyIHZlcmlmaWVkIGNvcGllcy4iIiIKICAgIHBlcnNpc3RlbnQ9UGF0aChwZXJzaXN0ZW50KTtiYXNlPXBlcnNpc3RlbnQvJ2NoZWNrcG9pbnRzJztiYXNlLm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgbmFtZT1mJ2Vwb2NoLXtpbnQodHJhaW5lci5lcG9jaCkrMTowNGR9JwogICAgZ2VuZXJhdGlvbj1iYXNlL25hbWU7Z2VuZXJhdGlvbi5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgZmlsZXM9e30KICAgIGZvciBrZXkgaW4gKCdsYXN0JywnYmVzdCcpOgogICAgICAgIHNvdXJjZT1QYXRoKGdldGF0dHIodHJhaW5lcixrZXkpKQogICAgICAgIGlmIHNvdXJjZS5leGlzdHMoKToKICAgICAgICAgICAgdGFyZ2V0PWdlbmVyYXRpb24vZid7a2V5fS5wdCc7YXRvbWljX2NvcHkoc291cmNlLHRhcmdldCk7ZmlsZXNba2V5XT1zaGEodGFyZ2V0KQogICAgaWYgJ2xhc3QnIG5vdCBpbiBmaWxlczpyYWlzZSBJT0Vycm9yKCdObyByZXN1bWFibGUgY2hlY2twb2ludCB3YXMgc2F2ZWQnKQogICAgcmVjb3JkPXsqKm1ldGFkYXRhLCdlcG9jaCc6aW50KHRyYWluZXIuZXBvY2gpKzEsJ2ZpbGVzJzpmaWxlcywnZ2VuZXJhdGlvbic6bmFtZX0KICAgIHRlbXBvcmFyeT1nZW5lcmF0aW9uLydzdGF0ZS5qc29uLnBhcnQnO3RlbXBvcmFyeS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVjb3JkLGluZGVudD0yKSk7dGVtcG9yYXJ5LnJlcGxhY2UoZ2VuZXJhdGlvbi8nc3RhdGUuanNvbicpCiAgICBwb2ludGVyPWJhc2UvJ2xhdGVzdC5qc29uLnBhcnQnO3BvaW50ZXIud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlY29yZCxpbmRlbnQ9MikpO3BvaW50ZXIucmVwbGFjZShiYXNlLydsYXRlc3QuanNvbicpCiAgICAjIE9ubHkgZ2VuZXJhdGVkIHByb2plY3QgY2hlY2twb2ludHMgdW5kZXIgdGhpcyBtb2RlIGRpcmVjdG9yeSBhcmUgZWxpZ2libGUgZm9yIHJvdGF0aW9uLgogICAgY29tcGxldGU9c29ydGVkKFtwIGZvciBwIGluIGJhc2UuZ2xvYignZXBvY2gtKicpIGlmIChwLydzdGF0ZS5qc29uJykuaXNfZmlsZSgpXSkKICAgIGZvciBvbGQgaW4gY29tcGxldGVbOi0yXTpzaHV0aWwucm10cmVlKG9sZCkKICAgIHJldHVybiByZWNvcmQKCgpkZWYgdmVyaWZ5X2xhdGVzdChwZXJzaXN0ZW50KToKICAgIGJhc2U9UGF0aChwZXJzaXN0ZW50KS8nY2hlY2twb2ludHMnCiAgICBzdGF0ZT1qc29uLmxvYWRzKChiYXNlLydsYXRlc3QuanNvbicpLnJlYWRfdGV4dCgpKQogICAgZ2VuZXJhdGlvbj1iYXNlL3N0YXRlWydnZW5lcmF0aW9uJ10KICAgIGZvciBrZXksZXhwZWN0ZWQgaW4gc3RhdGVbJ2ZpbGVzJ10uaXRlbXMoKToKICAgICAgICBpZiBzaGEoZ2VuZXJhdGlvbi9mJ3trZXl9LnB0JykhPWV4cGVjdGVkOnJhaXNlIFZhbHVlRXJyb3IoJ1NhdmVkIGNoZWNrcG9pbnQgZmFpbGVkIHJlY292ZXJ5IGNoZWNrc3VtJykKICAgIHJldHVybiBnZW5lcmF0aW9uLydsYXN0LnB0JyxzdGF0ZQoKCmRlZiBzbmFwc2hvdF90cmFja2luZyhkYXRhYmFzZSxhcnRpZmFjdF9kaXIscGVyc2lzdGVudCk6CiAgICBwZXJzaXN0ZW50PVBhdGgocGVyc2lzdGVudCk7cGVyc2lzdGVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIGlmIFBhdGgoZGF0YWJhc2UpLmV4aXN0cygpOgogICAgICAgIHRlbXBvcmFyeT1wZXJzaXN0ZW50LydtbGZsb3cuZGIucGFydCc7dGVtcG9yYXJ5LnVubGluayhtaXNzaW5nX29rPVRydWUpCiAgICAgICAgd2l0aCBzcWxpdGUzLmNvbm5lY3QoZGF0YWJhc2UpIGFzIHNyYyxzcWxpdGUzLmNvbm5lY3QodGVtcG9yYXJ5KSBhcyBkc3Q6c3JjLmJhY2t1cChkc3QpCiAgICAgICAgdGVtcG9yYXJ5LnJlcGxhY2UocGVyc2lzdGVudC8nbWxmbG93LmRiJykKICAgIGlmIFBhdGgoYXJ0aWZhY3RfZGlyKS5leGlzdHMoKToKICAgICAgICBzaHV0aWwuY29weXRyZWUoYXJ0aWZhY3RfZGlyLHBlcnNpc3RlbnQvJ21sYXJ0aWZhY3RzJyxkaXJzX2V4aXN0X29rPVRydWUpCgoKZGVmIHJ1bihyb290LHBlcnNpc3RlbnQsbW9kZT0nYmVuY2htYXJrJyxhcHByb3ZlX2Z1bGw9RmFsc2UscmVzdW1lPUZhbHNlLHJlY292ZXJ5X2JhY2tlbmQ9Tm9uZSk6CiAgICByb290LHBlcnNpc3RlbnQ9UGF0aChyb290KS5yZXNvbHZlKCksUGF0aChwZXJzaXN0ZW50KS5yZXNvbHZlKCkKICAgIG1hbmlmZXN0PXZlcmlmeV9idW5kbGUocm9vdCkKICAgIGV4cGVjdGVkX3Njb3BlPSdiZW5jaG1hcmsnIGlmIG1vZGU9PSdiZW5jaG1hcmsnIGVsc2UgJ3RyYWludmFsJwogICAgaWYgbWFuaWZlc3RbJ3Njb3BlJ10hPWV4cGVjdGVkX3Njb3BlOnJhaXNlIFZhbHVlRXJyb3IoJ1dyb25nIGJ1bmRsZSBzY29wZSBmb3IgcmVxdWVzdGVkIHJ1bicpCiAgICBpZiBtb2RlPT0ndHJhaW4nIGFuZCBub3QgYXBwcm92ZV9mdWxsOnJhaXNlIFZhbHVlRXJyb3IoJ0Z1bGwgdHJhaW5pbmcgYXBwcm92YWwgaXMgcmVxdWlyZWQ7IGJlbmNobWFyayBvbmx5IGlzIGN1cnJlbnRseSBhdXRob3JpemVkJykKICAgICMgVGhlIG1vdW50ZWQgcGF0aCBtdXN0IGJlIG91dHNpZGUgZXBoZW1lcmFsIC9jb250ZW50LiBBIG1vdW50LWxldmVsIGNoZWNrIGZvbGxvd3MgYmVsb3cuCiAgICBpZiByZWNvdmVyeV9iYWNrZW5kIGlzIE5vbmU6CiAgICAgICAgaWYgbm90IHN0cihwZXJzaXN0ZW50KS5zdGFydHN3aXRoKCcvY29udGVudC9kcml2ZS9NeURyaXZlLycpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVc2UgYSBwcm9qZWN0IGRpcmVjdG9yeSB1bmRlciBtb3VudGVkIEdvb2dsZSBEcml2ZSBNeURyaXZlJykKICAgICAgICBpZiBub3Qgb3MucGF0aC5pc21vdW50KCcvY29udGVudC9kcml2ZScpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdHb29nbGUgRHJpdmUgaXMgbm90IG1vdW50ZWQ7IHBlcnNpc3RlbnQgcmVjb3ZlcnkgY2Fubm90IGJlIHByb21pc2VkJykKICAgIGVsc2U6CiAgICAgICAgIyBBIHByb3ZpZGVyIGFkYXB0ZXIgbXVzdCBwcm92ZSByZW1vdGUgcGVyc2lzdGVuY2UgYmVmb3JlIHRyYWluaW5nOyBsb2NhbCBkaXNrIGlzIGluc3VmZmljaWVudC4KICAgICAgICByZWNvdmVyeV9iYWNrZW5kLnZhbGlkYXRlKHBlcnNpc3RlbnQsIG1vZGUsIHJlc3VtZSkKICAgIGltcG9ydCB5YW1sCiAgICBpbXBvcnQgdG9yY2gKICAgIGltcG9ydCBtbGZsb3cKICAgIGZyb20gdWx0cmFseXRpY3MgaW1wb3J0IFlPTE8sc2V0dGluZ3MKICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOnJhaXNlIFJ1bnRpbWVFcnJvcignTm8gQ1VEQSBHUFUgYWxsb2NhdGVkLiBTdG9wOyBuZXZlciBmYWxsIGJhY2sgdG8gTWFjL0NQVSBvciBhIHBhaWQgcnVudGltZSBhdXRvbWF0aWNhbGx5LicpCiAgICBwZXJzaXN0ZW50PXBlcnNpc3RlbnQvbW9kZTtwZXJzaXN0ZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgd3JpdGVfcHJvYmU9cGVyc2lzdGVudC8ncGVyc2lzdGVuY2UtcHJvYmUudHh0Jzt3cml0ZV9wcm9iZS53cml0ZV90ZXh0KCdtbG9wcy1jaGVja3BvaW50LXByb2JlJykKICAgIGlmIHdyaXRlX3Byb2JlLnJlYWRfdGV4dCgpIT0nbWxvcHMtY2hlY2twb2ludC1wcm9iZSc6cmFpc2UgSU9FcnJvcignUGVyc2lzdGVudCBzdG9yYWdlIGNoZWNrIGZhaWxlZCcpCiAgICB3b3JrPVBhdGgoJy9jb250ZW50L21sb3BzX3dvcmsnKS9tb2RlO3dvcmsubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBkYXRhYmFzZT13b3JrLydtbGZsb3cuZGInO2FydGlmYWN0X2Rpcj13b3JrLydtbGFydGlmYWN0cycKICAgIGlmIHJlc3VtZSBhbmQgbm90IGRhdGFiYXNlLmV4aXN0cygpIGFuZCAocGVyc2lzdGVudC8nbWxmbG93LmRiJykuZXhpc3RzKCk6CiAgICAgICAgc2h1dGlsLmNvcHkyKHBlcnNpc3RlbnQvJ21sZmxvdy5kYicsZGF0YWJhc2UpCiAgICAgICAgaWYgKHBlcnNpc3RlbnQvJ21sYXJ0aWZhY3RzJykuZXhpc3RzKCk6c2h1dGlsLmNvcHl0cmVlKHBlcnNpc3RlbnQvJ21sYXJ0aWZhY3RzJyxhcnRpZmFjdF9kaXIsZGlyc19leGlzdF9vaz1UcnVlKQogICAgbWxmbG93LnNldF90cmFja2luZ191cmkoZidzcWxpdGU6Ly8ve2RhdGFiYXNlfScpCiAgICBleHBlcmltZW50PW1sZmxvdy5nZXRfZXhwZXJpbWVudF9ieV9uYW1lKCdwbGFzdGljLXdhc3RlLXBoYXNlMS1jbG91ZCcpCiAgICBpZiBleHBlcmltZW50IGlzIE5vbmU6bWxmbG93LmNyZWF0ZV9leHBlcmltZW50KCdwbGFzdGljLXdhc3RlLXBoYXNlMS1jbG91ZCcsYXJ0aWZhY3RfbG9jYXRpb249YXJ0aWZhY3RfZGlyLmFzX3VyaSgpKQogICAgbWxmbG93LnNldF9leHBlcmltZW50KCdwbGFzdGljLXdhc3RlLXBoYXNlMS1jbG91ZCcpCiAgICBjb25maWc9eWFtbC5zYWZlX2xvYWQoKHJvb3QvJ2NvbmZpZy55YW1sJykucmVhZF90ZXh0KCkpO3BhcmFtcz1kaWN0KGNvbmZpZ1sndHJhaW4nXSkKICAgIHBhcmFtc1snbW9kZWwnXT1zdHIocm9vdC8nbW9kZWxzL3lvbG8xMW0ucHQnKTtwYXJhbXNbJ2RldmljZSddPScwJwogICAgaWYgbW9kZT09J2JlbmNobWFyayc6cGFyYW1zLnVwZGF0ZShlcG9jaHM9MixjbG9zZV9tb3NhaWM9MCxwbG90cz1GYWxzZSkKICAgIGRhdGE9eWFtbC5zYWZlX2xvYWQoKHJvb3QvJ2RhdGFzZXQvZGF0YXNldC55YW1sJykucmVhZF90ZXh0KCkpO2RhdGFbJ3BhdGgnXT1zdHIocm9vdC8nZGF0YXNldCcpCiAgICBydW50aW1lPXdvcmsvJ2RhdGFzZXQueWFtbCc7cnVudGltZS53cml0ZV90ZXh0KHlhbWwuc2FmZV9kdW1wKGRhdGEpKQogICAgc2V0dGluZ3MudXBkYXRlKHtrZXk6RmFsc2UgZm9yIGtleSBpbiAoJ21sZmxvdycsJ3dhbmRiJywnY2xlYXJtbCcsJ2NvbWV0JywnbmVwdHVuZScsJ3RlbnNvcmJvYXJkJykgaWYga2V5IGluIHNldHRpbmdzfSkKICAgIG9zLmVudmlyb25bJ01MRkxPV19ESVNBQkxFX0FHRU5UX0hJTlQnXT0nMScKICAgIGZyZWV6ZT1zdWJwcm9jZXNzLmNoZWNrX291dHB1dChbb3Muc3lzLmV4ZWN1dGFibGUsJy1tJywncGlwJywnZnJlZXplJ10sdGV4dD1UcnVlKQogICAgKHdvcmsvJ3JlcXVpcmVtZW50cy1jbG91ZC1yZXNvbHZlZC50eHQnKS53cml0ZV90ZXh0KGZyZWV6ZSkKICAgIGdwdT17J25hbWUnOnRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLCd2cmFtX0dpQic6dG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoMCkudG90YWxfbWVtb3J5LzEwMjQqKjMsCiAgICAgICAgJ3RvcmNoJzp0b3JjaC5fX3ZlcnNpb25fXywnY3VkYSc6dG9yY2gudmVyc2lvbi5jdWRhLCdweXRob24nOnBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCl9CiAgICAod29yay8nZ3B1Lmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZ3B1LGluZGVudD0yKSkKICAgIGNoZWNrcG9pbnQ9Tm9uZTtydW5faWQ9Tm9uZQogICAgZmluZ2VycHJpbnQ9c2hhKHJvb3QvJ21hbmlmZXN0Lmpzb24nKQogICAgaWYgcmVzdW1lOgogICAgICAgIGNoZWNrcG9pbnQsc3RhdGU9dmVyaWZ5X2xhdGVzdChwZXJzaXN0ZW50KQogICAgICAgIGlmIHN0YXRlWydtb2RlJ10hPW1vZGUgb3Igc3RhdGVbJ2J1bmRsZV9maW5nZXJwcmludCddIT1maW5nZXJwcmludDpyYWlzZSBWYWx1ZUVycm9yKCdSZWNvdmVyeSBidW5kbGUvbW9kZSBtaXNtYXRjaCcpCiAgICAgICAgcnVuX2lkPXN0YXRlWydydW5faWQnXQogICAgICAgIHJlc3RvcmVkPXdvcmsvJ3J1bnMnL3J1bl9pZDtyZXN0b3JlZC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBpZiAocGVyc2lzdGVudC8ndHJhaW5pbmcnKS5leGlzdHMoKTpzaHV0aWwuY29weXRyZWUocGVyc2lzdGVudC8ndHJhaW5pbmcnLHJlc3RvcmVkLGRpcnNfZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBmb3Iga2V5IGluIHN0YXRlWydmaWxlcyddOgogICAgICAgICAgICBhdG9taWNfY29weShjaGVja3BvaW50LnBhcmVudC9mJ3trZXl9LnB0JyxyZXN0b3JlZC8nd2VpZ2h0cycvZid7a2V5fS5wdCcpCiAgICB3aXRoIG1sZmxvdy5zdGFydF9ydW4ocnVuX2lkPXJ1bl9pZCxydW5fbmFtZT1Ob25lIGlmIHJlc3VtZSBlbHNlIGYneW9sbzExbS17bW9kZX0te3RpbWUuc3RyZnRpbWUoIiVZJW0lZC0lSCVNJVMiKX0nKSBhcyBleHBlcmltZW50X3J1bjoKICAgICAgICBydW5faWQ9ZXhwZXJpbWVudF9ydW4uaW5mby5ydW5faWQKICAgICAgICBtbGZsb3cuc2V0X3RhZ3MoeydwaGFzZSc6JzEnLCdwcm9maWxlJzpmJ2Nsb3VkLXttb2RlfScsJ2RhdGFzZXRfYXBwcm92YWwnOidwZW5kaW5nJywnYnVuZGxlX2ZpbmdlcnByaW50JzpmaW5nZXJwcmludCwKICAgICAgICAgICAgJ3NvdXJjZV9naXRfY29tbWl0JzptYW5pZmVzdFsnc291cmNlX2dpdF9jb21taXQnXSwnbm90X2Zvcl9xdWFsaXR5X2NsYWltcyc6c3RyKG1vZGU9PSdiZW5jaG1hcmsnKSwnZ3B1JzpncHVbJ25hbWUnXX0pCiAgICAgICAgaWYgbm90IHJlc3VtZTptbGZsb3cubG9nX3BhcmFtcyhwYXJhbXMpCiAgICAgICAgZm9yIHBhdGggaW4gW3Jvb3QvJ21hbmlmZXN0Lmpzb24nLHJvb3QvJ2NvbmZpZy55YW1sJyx3b3JrLydyZXF1aXJlbWVudHMtY2xvdWQtcmVzb2x2ZWQudHh0Jyx3b3JrLydncHUuanNvbiddOgogICAgICAgICAgICBtbGZsb3cubG9nX2FydGlmYWN0KHN0cihwYXRoKSxhcnRpZmFjdF9wYXRoPSdwcm92ZW5hbmNlJykKICAgICAgICB0b3JjaC5jdWRhLnJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKCkKICAgICAgICBtb2RlbD1ZT0xPKHN0cihjaGVja3BvaW50KSBpZiBjaGVja3BvaW50IGVsc2UgcGFyYW1zWydtb2RlbCddKQogICAgICAgIG1ldGFkYXRhPXsnbW9kZSc6bW9kZSwncnVuX2lkJzpydW5faWQsJ2J1bmRsZV9maW5nZXJwcmludCc6ZmluZ2VycHJpbnR9CiAgICAgICAgc2F2ZWQ9W107dGltZXM9W107c3RhcnQ9W05vbmVdCiAgICAgICAgZGVmIGJhdGNoX3N0YXJ0KHRyYWluZXIpOgogICAgICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCk7c3RhcnRbMF09dGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIGRlZiBiYXRjaF9lbmQodHJhaW5lcik6CiAgICAgICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTt0aW1lcy5hcHBlbmQodGltZS5wZXJmX2NvdW50ZXIoKS1zdGFydFswXSkKICAgICAgICBkZWYgb25fc2F2ZSh0cmFpbmVyKToKICAgICAgICAgICAgc3RhdGU9c2F2ZV9jaGVja3BvaW50X2dlbmVyYXRpb24odHJhaW5lcixwZXJzaXN0ZW50LG1ldGFkYXRhKTtzYXZlZC5hcHBlbmQoc3RhdGUpCiAgICAgICAgICAgIGZvciBuYW1lIGluICgncmVzdWx0cy5jc3YnLCdhcmdzLnlhbWwnKToKICAgICAgICAgICAgICAgIHA9UGF0aCh0cmFpbmVyLnNhdmVfZGlyKS9uYW1lCiAgICAgICAgICAgICAgICBpZiBwLmV4aXN0cygpOmF0b21pY19jb3B5KHAscGVyc2lzdGVudC8ndHJhaW5pbmcnL25hbWUpCiAgICAgICAgICAgIG1sZmxvdy5sb2dfbWV0cmljcyh7ay5yZXBsYWNlKCcoJywnXycpLnJlcGxhY2UoJyknLCdfJyk6ZmxvYXQodikgZm9yIGssdiBpbiB0cmFpbmVyLm1ldHJpY3MuaXRlbXMoKX0sc3RlcD1pbnQodHJhaW5lci5lcG9jaCkpCiAgICAgICAgICAgIHNuYXBzaG90X3RyYWNraW5nKGRhdGFiYXNlLGFydGlmYWN0X2RpcixwZXJzaXN0ZW50KQogICAgICAgICAgICBpZiByZWNvdmVyeV9iYWNrZW5kIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgcmVjb3ZlcnlfYmFja2VuZC5wdWJsaXNoKHBlcnNpc3RlbnQpCiAgICAgICAgbW9kZWwuYWRkX2NhbGxiYWNrKCdvbl90cmFpbl9iYXRjaF9zdGFydCcsYmF0Y2hfc3RhcnQpO21vZGVsLmFkZF9jYWxsYmFjaygnb25fdHJhaW5fYmF0Y2hfZW5kJyxiYXRjaF9lbmQpCiAgICAgICAgbW9kZWwuYWRkX2NhbGxiYWNrKCdvbl9tb2RlbF9zYXZlJyxvbl9zYXZlKQogICAgICAgIHN0YXJ0ZWQ9dGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIHRyeToKICAgICAgICAgICAgaWYgcmVzdW1lOnJlc3VsdD1tb2RlbC50cmFpbihyZXN1bWU9VHJ1ZSxkZXZpY2U9JzAnKQogICAgICAgICAgICBlbHNlOnJlc3VsdD1tb2RlbC50cmFpbihkYXRhPXN0cihydW50aW1lKSxwcm9qZWN0PXN0cih3b3JrLydydW5zJyksbmFtZT1ydW5faWQsZXhpc3Rfb2s9RmFsc2Usc2F2ZV9wZXJpb2Q9LTEsKip7azp2IGZvciBrLHYgaW4gcGFyYW1zLml0ZW1zKCkgaWYgayE9J21vZGVsJ30pCiAgICAgICAgICAgIGxhc3Qsc3RhdGU9dmVyaWZ5X2xhdGVzdChwZXJzaXN0ZW50KQogICAgICAgICAgICAjIFZlcmlmeSB0aGUgcmVjb3ZlcnkgcGF5bG9hZCBhY3R1YWxseSBjb250YWlucyBlcG9jaCArIG9wdGltaXplciBzdGF0ZSwgd2l0aG91dCBsb2FkaW5nIENVREEgdGVuc29ycy4KICAgICAgICAgICAgcGF5bG9hZD10b3JjaC5sb2FkKGxhc3QsbWFwX2xvY2F0aW9uPSdjcHUnLHdlaWdodHNfb25seT1GYWxzZSkKICAgICAgICAgICAgcmVjb3ZlcmFibGU9cGF5bG9hZC5nZXQoJ29wdGltaXplcicpIGlzIG5vdCBOb25lIGFuZCBwYXlsb2FkLmdldCgnZXBvY2gnLC0xKT49MAogICAgICAgICAgICBkZWwgcGF5bG9hZAogICAgICAgICAgICByZWNvcmQ9eyoqbWV0YWRhdGEsJ3N0YXR1cyc6J2NvbXBsZXRlZCcsJ2dwdSc6Z3B1LCdlbGFwc2VkX3NlY29uZHMnOnRpbWUucGVyZl9jb3VudGVyKCktc3RhcnRlZCwKICAgICAgICAgICAgICAgICdwZWFrX2N1ZGFfYWxsb2NhdGVkX0dpQic6dG9yY2guY3VkYS5tYXhfbWVtb3J5X2FsbG9jYXRlZCgpLzEwMjQqKjMsCiAgICAgICAgICAgICAgICAncGVha19jdWRhX3Jlc2VydmVkX0dpQic6dG9yY2guY3VkYS5tYXhfbWVtb3J5X3Jlc2VydmVkKCkvMTAyNCoqMywKICAgICAgICAgICAgICAgICdtZWFuX2JhdGNoX2NvbXB1dGVfc2Vjb25kc19hZnRlcjEwJzpzdW0odGltZXNbMTA6XSkvbWF4KDEsbGVuKHRpbWVzWzEwOl0pKSwKICAgICAgICAgICAgICAgICdwZXJzaXN0ZW50X2NoZWNrcG9pbnRfdmVyaWZpZWQnOlRydWUsJ29wdGltaXplcl9zdGF0ZV92ZXJpZmllZCc6cmVjb3ZlcmFibGUsCiAgICAgICAgICAgICAgICAnc2F2ZWRfZ2VuZXJhdGlvbnMnOmxlbihzYXZlZCksJ2RhdGFzZXRfaW1hZ2VzJzptYW5pZmVzdFsndHJhaW5faW1hZ2VzJ10sCiAgICAgICAgICAgICAgICAncXVhbGl0eV9jbGFpbSc6RmFsc2UgaWYgbW9kZT09J2JlbmNobWFyaycgZWxzZSAnVmFsaWRhdGlvbiBvbmx5OyBzZXBhcmF0ZSBmaW5hbCBldmFsdWF0aW9uIHJlcXVpcmVkJywKICAgICAgICAgICAgICAgICdub3RlJzonQmF0Y2ggdGltaW5nIGV4Y2x1ZGVzIGxvYWRlciB0aW1lOyB1c2UgZXBvY2ggcmVzdWx0cy5jc3YgZm9yIGZ1bGwtcnVuIEVUQS4gTm8gNzAlIHBlcmZvcm1hbmNlIGNsYWltLid9CiAgICAgICAgICAgIGlmIG1vZGU9PSdiZW5jaG1hcmsnIGFuZCBub3QgcmVjb3ZlcmFibGU6cmFpc2UgUnVudGltZUVycm9yKCdCZW5jaG1hcmsgcmVjb3ZlcnkgcGF5bG9hZCBsYWNrcyBvcHRpbWl6ZXIgc3RhdGUnKQogICAgICAgICAgICBvdXRwdXQ9d29yay9mJ3ttb2RlfV9yZXBvcnQuanNvbic7b3V0cHV0LndyaXRlX3RleHQoanNvbi5kdW1wcyhyZWNvcmQsaW5kZW50PTIpKQogICAgICAgICAgICBhdG9taWNfY29weShvdXRwdXQscGVyc2lzdGVudC9vdXRwdXQubmFtZSk7bWxmbG93LmxvZ19hcnRpZmFjdChzdHIob3V0cHV0KSkKICAgICAgICAgICAgaWYgbW9kZT09J3RyYWluJzoKICAgICAgICAgICAgICAgIGRlc3RpbmF0aW9uPXdvcmsvJ3lvbG8xMW0td2FzdGUtYmVzdC5wdCc7c2h1dGlsLmNvcHkyKG1vZGVsLnRyYWluZXIuYmVzdCxkZXN0aW5hdGlvbikKICAgICAgICAgICAgICAgIGF0b21pY19jb3B5KGRlc3RpbmF0aW9uLHBlcnNpc3RlbnQvZGVzdGluYXRpb24ubmFtZSk7bWxmbG93LmxvZ19hcnRpZmFjdChzdHIoZGVzdGluYXRpb24pLGFydGlmYWN0X3BhdGg9J3dlaWdodHMnKQogICAgICAgICAgICBmb3IgcCBpbiBQYXRoKG1vZGVsLnRyYWluZXIuc2F2ZV9kaXIpLml0ZXJkaXIoKToKICAgICAgICAgICAgICAgIGlmIHAuc3VmZml4IGluICgnLmNzdicsJy55YW1sJywnLnBuZycsJy5qcGcnKToKICAgICAgICAgICAgICAgICAgICBtbGZsb3cubG9nX2FydGlmYWN0KHN0cihwKSxhcnRpZmFjdF9wYXRoPSd0cmFpbmluZycpCiAgICAgICAgICAgICAgICAgICAgYXRvbWljX2NvcHkocCxwZXJzaXN0ZW50Lyd0cmFpbmluZycvcC5uYW1lKQogICAgICAgICAgICBwcmludChqc29uLmR1bXBzKHJlY29yZCxpbmRlbnQ9MikpCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgIyBTUUxpdGUgc25hcHNob3RzIHVzZSBpdHMgYmFja3VwIEFQSSByYXRoZXIgdGhhbiBjb3B5aW5nIGEgbGl2ZSBXQUwgZGF0YWJhc2UuCiAgICAgICAgICAgIHNuYXBzaG90X3RyYWNraW5nKGRhdGFiYXNlLGFydGlmYWN0X2RpcixwZXJzaXN0ZW50KQogICAgc25hcHNob3RfdHJhY2tpbmcoZGF0YWJhc2UsYXJ0aWZhY3RfZGlyLHBlcnNpc3RlbnQpCiAgICBpZiByZWNvdmVyeV9iYWNrZW5kIGlzIG5vdCBOb25lOgogICAgICAgIHJlY292ZXJ5X2JhY2tlbmQucHVibGlzaChwZXJzaXN0ZW50KQoKCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOgogICAgcGFyc2VyPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7cGFyc2VyLmFkZF9hcmd1bWVudCgnbW9kZScsY2hvaWNlcz1bJ2JlbmNobWFyaycsJ3RyYWluJ10pO3BhcnNlci5hZGRfYXJndW1lbnQoJy0tcm9vdCcscmVxdWlyZWQ9VHJ1ZSk7cGFyc2VyLmFkZF9hcmd1bWVudCgnLS1wZXJzaXN0ZW50JyxyZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1hcHByb3ZlLWZ1bGwnLGFjdGlvbj0nc3RvcmVfdHJ1ZScpO3BhcnNlci5hZGRfYXJndW1lbnQoJy0tcmVzdW1lJyxhY3Rpb249J3N0b3JlX3RydWUnKQogICAgYXJncz1wYXJzZXIucGFyc2VfYXJncygpO3J1bihhcmdzLnJvb3QsYXJncy5wZXJzaXN0ZW50LGFyZ3MubW9kZSxhcmdzLmFwcHJvdmVfZnVsbCxhcmdzLnJlc3VtZSkK", "verify_migration.py": "IiIiUmVhZC1vbmx5IGNoZWNrcyBiZWZvcmUgbW92aW5nIHRoZSBleGlzdGluZyBjbG91ZCBleHBlcmltZW50IGJldHdlZW4gcHJvdmlkZXJzLiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzcWxpdGUzCmZyb20gcnVubmVyIGltcG9ydCBzaGEKClJVTl9JRCA9ICc4NTcxNDQ2Njk0OGQ0ZTY1OGViYjYxMGE0NzVkY2Y1NycKRVhQRUNURURfVkVSU0lPTlMgPSB7J3RvcmNoJzogJzIuMTEuMCtjdTEzMCcsICd0b3JjaHZpc2lvbic6ICcwLjI2LjArY3UxMzAnLAogICAgICAgICAgICAgICAgICAgICAndWx0cmFseXRpY3MnOiAnOC40LjIwMScsICdtbGZsb3cnOiAnMy4xNy4yJ30KCgpkZWYgdmVyaWZ5X3JlY292ZXJ5KHRyYWluX2RpcmVjdG9yeSwgbWFuaWZlc3RfcGF0aCwgZXhwZWN0ZWRfcnVuPVJVTl9JRCwgbWluaW11bV9lcG9jaD0zMCk6CiAgICB0cmFpbiA9IFBhdGgodHJhaW5fZGlyZWN0b3J5KQogICAgcG9pbnRlciA9IGpzb24ubG9hZHMoKHRyYWluIC8gJ2NoZWNrcG9pbnRzL2xhdGVzdC5qc29uJykucmVhZF90ZXh0KCkpCiAgICBlcG9jaCA9IHBvaW50ZXJbJ2Vwb2NoJ10KICAgIGlmIHR5cGUoZXBvY2gpIGlzIG5vdCBpbnQgb3IgZXBvY2ggPCBtaW5pbXVtX2Vwb2NoOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1JlY292ZXJ5IHdvdWxkIHJvbGwgYmFjayBjb21wbGV0ZWQgZXBvY2hzJykKICAgIGlmIHBvaW50ZXJbJ3J1bl9pZCddICE9IGV4cGVjdGVkX3J1biBvciBwb2ludGVyWydtb2RlJ10gIT0gJ3RyYWluJzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdXcm9uZyBleHBlcmltZW50IG9yIHJlY292ZXJ5IG1vZGUnKQogICAgaWYgcG9pbnRlclsnZ2VuZXJhdGlvbiddICE9IGYnZXBvY2gte2Vwb2NoOjA0ZH0nOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1Vuc2FmZSBvciBpbmNvbnNpc3RlbnQgY2hlY2twb2ludCBnZW5lcmF0aW9uJykKICAgIGlmIHBvaW50ZXJbJ2J1bmRsZV9maW5nZXJwcmludCddICE9IHNoYShQYXRoKG1hbmlmZXN0X3BhdGgpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdEYXRhc2V0IGJ1bmRsZSBmaW5nZXJwcmludCBjaGFuZ2VkJykKICAgIGlmIHNldChwb2ludGVyWydmaWxlcyddKSAhPSB7J2xhc3QnLCAnYmVzdCd9OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0JvdGggbGFzdCBhbmQgYmVzdCB3ZWlnaHRzIGFyZSByZXF1aXJlZCcpCiAgICBnZW5lcmF0aW9uID0gdHJhaW4gLyAnY2hlY2twb2ludHMnIC8gcG9pbnRlclsnZ2VuZXJhdGlvbiddCiAgICBpZiBqc29uLmxvYWRzKChnZW5lcmF0aW9uIC8gJ3N0YXRlLmpzb24nKS5yZWFkX3RleHQoKSkgIT0gcG9pbnRlcjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdDaGVja3BvaW50IGdlbmVyYXRpb24gbWV0YWRhdGEgZGlmZmVycyBmcm9tIGxhdGVzdCBwb2ludGVyJykKICAgIGZvciBrZXksIGRpZ2VzdCBpbiBwb2ludGVyWydmaWxlcyddLml0ZW1zKCk6CiAgICAgICAgaWYgc2hhKGdlbmVyYXRpb24gLyBmJ3trZXl9LnB0JykgIT0gZGlnZXN0OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdDaGVja3BvaW50IGhhc2ggbWlzbWF0Y2gnKQogICAgZGF0YWJhc2UgPSB0cmFpbiAvICdtbGZsb3cuZGInCiAgICB3aXRoIHNxbGl0ZTMuY29ubmVjdChkYXRhYmFzZS5yZXNvbHZlKCkuYXNfdXJpKCkgKyAnP21vZGU9cm8nLCB1cmk9VHJ1ZSkgYXMgZGI6CiAgICAgICAgaWYgZGIuZXhlY3V0ZSgnUFJBR01BIGludGVncml0eV9jaGVjaycpLmZldGNob25lKClbMF0gIT0gJ29rJzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVHJhY2tpbmcgZGF0YWJhc2UgaW50ZWdyaXR5IGNoZWNrIGZhaWxlZCcpCiAgICAgICAgcm93ID0gZGIuZXhlY3V0ZSgnU0VMRUNUIHJ1bl91dWlkLCBhcnRpZmFjdF91cmkgRlJPTSBydW5zIFdIRVJFIHJ1bl91dWlkID0gPycsIChleHBlY3RlZF9ydW4sKSkuZmV0Y2hvbmUoKQogICAgICAgIGlmIHJvdyBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdTYW1lIE1MZmxvdyBydW4gaXMgbWlzc2luZyBmcm9tIHJlY292ZXJ5IGRhdGFiYXNlJykKICAgICAgICBhcnRpZmFjdF91cmkgPSByb3dbMV0KICAgIGZvciBmaWxlbmFtZSBpbiAoJ2FyZ3MueWFtbCcsICdyZXN1bHRzLmNzdicpOgogICAgICAgIGlmIG5vdCAodHJhaW4gLyAndHJhaW5pbmcnIC8gZmlsZW5hbWUpLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVHJhaW5pbmcgaGlzdG9yeS9jb25maWd1cmF0aW9uIGlzIG1pc3NpbmcnKQogICAgaWYgbm90ICh0cmFpbiAvICdtbGFydGlmYWN0cycpLmlzX2RpcigpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1RyYWNraW5nIGFydGlmYWN0cyBhcmUgbWlzc2luZycpCiAgICByZXR1cm4geydydW5faWQnOiBleHBlY3RlZF9ydW4sICdlcG9jaCc6IGVwb2NoLCAnY2hlY2twb2ludF9zaGEyNTYnOiBwb2ludGVyWydmaWxlcyddLAogICAgICAgICAgICAnYnVuZGxlX2ZpbmdlcnByaW50JzogcG9pbnRlclsnYnVuZGxlX2ZpbmdlcnByaW50J10sICdzcWxpdGVfaW50ZWdyaXR5JzogJ29rJywKICAgICAgICAgICAgJ29yaWdpbmFsX2FydGlmYWN0X3VyaSc6IGFydGlmYWN0X3VyaSwgJ3dlaWdodHNfYW5kX3RyYWNraW5nX3ZlcmlmaWVkJzogVHJ1ZSwKICAgICAgICAgICAgJ29wdGltaXplcl9zdGF0ZV92ZXJpZmllZCc6IEZhbHNlLAogICAgICAgICAgICAnbm90ZSc6ICdMb2FkIGNoZWNrcG9pbnQgd2l0aCB0aGUgcGlubmVkIFRvcmNoIGVudmlyb25tZW50IGFuZCB2ZXJpZnkgb3B0aW1pemVyL2Vwb2NoIGJlZm9yZSB0cmFpbmluZy4gTm8gdHJhaW5pbmcgcGVyZm9ybWVkLid9CgoKZGVmIHZlcmlmeV92ZXJzaW9ucyhhY3R1YWwpOgogICAgaWYgYWN0dWFsICE9IEVYUEVDVEVEX1ZFUlNJT05TOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZidFeGFjdCB0cmFpbmluZyBkZXBlbmRlbmNpZXMgcmVxdWlyZWQ6IHtFWFBFQ1RFRF9WRVJTSU9OU307IGZvdW5kIHthY3R1YWx9JykKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS10cmFpbi1kaXJlY3RvcnknLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1tYW5pZmVzdCcsIHJlcXVpcmVkPVRydWUpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcHJpbnQoanNvbi5kdW1wcyh2ZXJpZnlfcmVjb3ZlcnkoYXJncy50cmFpbl9kaXJlY3RvcnksIGFyZ3MubWFuaWZlc3QpLCBpbmRlbnQ9MikpCg==", "kaggle_preflight.py": "IiIiVmVyaWZ5IHRoZSBtaWdyYXRlZCBleHBlcmltZW50IG9uIEthZ2dsZSB3aXRob3V0IHN0YXJ0aW5nIHRyYWluaW5nLiIiIgppbXBvcnQgaW1wb3J0bGliLm1ldGFkYXRhCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc2h1dGlsCmltcG9ydCBzdGF0CmltcG9ydCB6aXBmaWxlCmZyb20gY29uY3VycmVudC5mdXR1cmVzIGltcG9ydCBUaHJlYWRQb29sRXhlY3V0b3IKCmltcG9ydCB0b3JjaApmcm9tIHJ1bm5lciBpbXBvcnQgc2hhLCB1bnBhY2ssIHZlcmlmeV9idW5kbGUKZnJvbSB2ZXJpZnlfbWlncmF0aW9uIGltcG9ydCBFWFBFQ1RFRF9WRVJTSU9OUywgdmVyaWZ5X3JlY292ZXJ5LCB2ZXJpZnlfdmVyc2lvbnMKCkFSQ0hJVkVfU0hBID0gJzM3YzdlNTBiOTU0YTk5NjUxOWZjOGQ0ZjUyNzE1ZmM5M2M3MzNiMzEzMzc0YWI0ZjJiOTFhZjk3MjUwNjQwOTgnClJFQ09WRVJZX1NIQSA9ICc4YmZkZWYxMmE3OTE4Y2NiNDUyYTU5MDgxNWZjYjI4MDRmYWEzMzM2MjZkNjA4Yjc1YWVjYzg0OWU0MzU2OTQ5JwoKCmRlZiBidWZmZXJlZF9jb3B5KHNvdXJjZSwgZGVzdGluYXRpb24pOgogICAgIyBBdm9pZCBzZW5kZmlsZSBvbiBLYWdnbGUncyBpbnB1dCBmaWxlc3lzdGVtOyB1c2UgYm91bmRlZCBidWZmZXJlZCByZWFkcy4KICAgIHdpdGggb3Blbihzb3VyY2UsICdyYicpIGFzIHNyYywgb3BlbihkZXN0aW5hdGlvbiwgJ3diJykgYXMgZHN0OgogICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihzcmMsIGRzdCwgbGVuZ3RoPTEwMjQgKiAxMDI0KQogICAgcmV0dXJuIHN0cihkZXN0aW5hdGlvbikKCgpkZWYgaW5wdXRfcm9vdChpbnB1dHMsIHNsdWcpOgogICAgY2FuZGlkYXRlcyA9IFtpbnB1dHMgLyAnZGF0YXNldHMvYWF5dXNoYm1sb3BzMjYnIC8gc2x1ZywgaW5wdXRzIC8gc2x1Z10KICAgIGZvdW5kID0gW3AgZm9yIHAgaW4gY2FuZGlkYXRlcyBpZiBwLmlzX2RpcigpXQogICAgaWYgbGVuKGZvdW5kKSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZidFeHBlY3RlZCBvbmUgYXR0YWNoZWQgaW5wdXQgYXQgaXRzIGtub3duIG1vdW50IHBhdGg6IHtzbHVnfScpCiAgICByZXR1cm4gZm91bmRbMF0KCgpkZWYgbWFpbigpOgogICAgYXNzZXJ0IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCksICdBIGZyZWUgQ1VEQSBHUFUgaXMgcmVxdWlyZWQuJwogICAgYWN0dWFsID0ge25hbWU6IGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKG5hbWUpIGZvciBuYW1lIGluIEVYUEVDVEVEX1ZFUlNJT05TfQogICAgdmVyaWZ5X3ZlcnNpb25zKGFjdHVhbCkKICAgIGlucHV0cyA9IFBhdGgoJy9rYWdnbGUvaW5wdXQnKQogICAgZGF0YV9pbnB1dCA9IGlucHV0X3Jvb3QoaW5wdXRzLCAnbWxvcHMtZWwtbGFiZWxzLXYzLXRyYWludmFsLXByaXZhdGUnKQogICAgcmVjb3ZlcnlfaW5wdXQgPSBpbnB1dF9yb290KGlucHV0cywgJ21sb3BzLWVsLWVwb2NoMzAtcmVjb3ZlcnktcHJpdmF0ZScpCiAgICB3b3JrID0gUGF0aCgnL2thZ2dsZS93b3JraW5nL21sb3BzX21pZ3JhdGlvbicpCiAgICB3b3JrLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHJvb3QgPSBQYXRoKCcva2FnZ2xlL3RlbXAvbWxvcHNfdHJhaW52YWxfYnVuZGxlJykKICAgIHByaW50KCdTdGFnaW5nIGZ1bGwgdHJhaW4vdmFsaWRhdGlvbiBpbnB1dCBmcm9tIGtub3duIG1vdW50IHBhdGguJywgZmx1c2g9VHJ1ZSkKICAgIGFyY2hpdmUgPSBkYXRhX2lucHV0IC8gJ21sb3BzLXRyYWludmFsLWxhYmVscy12My50YXInCiAgICBpZiBhcmNoaXZlLmlzX2ZpbGUoKToKICAgICAgICBpZiBub3Qgcm9vdC5leGlzdHMoKToKICAgICAgICAgICAgdW5wYWNrKGFyY2hpdmUsIHJvb3QsIEFSQ0hJVkVfU0hBKQogICAgZWxzZToKICAgICAgICBzb3VyY2UgPSBkYXRhX2lucHV0CiAgICAgICAgYXNzZXJ0IChzb3VyY2UgLyAnZGF0YXNldC9pbWFnZXMvdHJhaW4nKS5pc19kaXIoKSwgJ0NvbXBsZXRlIGlucHV0IGxheW91dCBpcyByZXF1aXJlZC4nCiAgICAgICAgbWFuaWZlc3QgPSBqc29uLmxvYWRzKChzb3VyY2UgLyAnbWFuaWZlc3QuanNvbicpLnJlYWRfdGV4dCgpKQogICAgICAgIGNhbm9uaWNhbCA9IGpzb24uZHVtcHMobWFuaWZlc3QsIGluZGVudD0yKQogICAgICAgIHJvb3QubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIHRhcmdldF9tYW5pZmVzdCA9IHJvb3QgLyAnbWFuaWZlc3QuanNvbicKICAgICAgICBpZiB0YXJnZXRfbWFuaWZlc3QuZXhpc3RzKCkgYW5kIHRhcmdldF9tYW5pZmVzdC5yZWFkX3RleHQoKSAhPSBjYW5vbmljYWw6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0FuIGV4aXN0aW5nIHN0YWdpbmcgZGlyZWN0b3J5IGJlbG9uZ3MgdG8gYSBkaWZmZXJlbnQgYnVuZGxlLicpCiAgICAgICAgZnJvbSBydW5uZXIgaW1wb3J0IHNhZmVfbmFtZQogICAgICAgIGRlZiBjb3B5X3BheWxvYWQocm93KToKICAgICAgICAgICAgcmVsYXRpdmUgPSBzYWZlX25hbWUocm93WydwYXRoJ10pCiAgICAgICAgICAgIHNyYywgZHN0ID0gc291cmNlLmpvaW5wYXRoKCpyZWxhdGl2ZS5wYXJ0cyksIHJvb3Quam9pbnBhdGgoKnJlbGF0aXZlLnBhcnRzKQogICAgICAgICAgICBpZiBub3QgZHN0LmlzX2ZpbGUoKSBvciBkc3Quc3RhdCgpLnN0X3NpemUgIT0gcm93WydieXRlcyddOgogICAgICAgICAgICAgICAgZHN0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgICAgICAgICBidWZmZXJlZF9jb3B5KHNyYywgZHN0KQogICAgICAgIHdpdGggVGhyZWFkUG9vbEV4ZWN1dG9yKG1heF93b3JrZXJzPTgpIGFzIHBvb2w6CiAgICAgICAgICAgIGZvciBjb3VudCwgXyBpbiBlbnVtZXJhdGUocG9vbC5tYXAoY29weV9wYXlsb2FkLCBtYW5pZmVzdFsnZmlsZXMnXSksIDEpOgogICAgICAgICAgICAgICAgaWYgY291bnQgJSAxMDAwID09IDA6CiAgICAgICAgICAgICAgICAgICAgcHJpbnQoZidTdGFnZWQge2NvdW50fS97bGVuKG1hbmlmZXN0WyJmaWxlcyJdKX0gcGF5bG9hZCBmaWxlcycsIGZsdXNoPVRydWUpCiAgICAgICAgdGFyZ2V0X21hbmlmZXN0LndyaXRlX3RleHQoY2Fub25pY2FsKQogICAgcHJpbnQoJ1ZlcmlmeWluZyBhbGwgcGF5bG9hZCBzaXplcywgaGFzaGVzIGFuZCBzcGxpdCBjb3VudHMuJywgZmx1c2g9VHJ1ZSkKICAgIHZlcmlmeV9idW5kbGUocm9vdCkKICAgIHRyYWluID0gd29yayAvICd0cmFpbicKICAgIHNvdXJjZV90cmFpbiA9IHJlY292ZXJ5X2lucHV0IC8gJ3RyYWluJwogICAgaWYgc291cmNlX3RyYWluLmlzX2RpcigpOgogICAgICAgIHByaW50KCdDb3B5aW5nIGltbXV0YWJsZSByZWNvdmVyeSBpbnB1dCB3aXRoIGJ1ZmZlcmVkIEkvTy4nLCBmbHVzaD1UcnVlKQogICAgICAgIHNodXRpbC5jb3B5dHJlZShzb3VyY2VfdHJhaW4sIHRyYWluLCBkaXJzX2V4aXN0X29rPVRydWUsIGNvcHlfZnVuY3Rpb249YnVmZmVyZWRfY29weSkKICAgIGVsaWYgbm90IHRyYWluLmV4aXN0cygpOgogICAgICAgIHJlY292ZXJ5X3ppcCA9IHJlY292ZXJ5X2lucHV0IC8gJ3RyYWluLTIwMjYxMTA2VDEwNTU1MlotMS0wMDEuemlwJwogICAgICAgIGlmIHJlY292ZXJ5X3ppcC5pc19maWxlKCk6CiAgICAgICAgICAgIGFzc2VydCBzaGEocmVjb3ZlcnlfemlwKSA9PSBSRUNPVkVSWV9TSEEsICdSZWNvdmVyeSBhcmNoaXZlIGNoZWNrc3VtIG1pc21hdGNoLicKICAgICAgICAgICAgd2l0aCB6aXBmaWxlLlppcEZpbGUocmVjb3ZlcnlfemlwKSBhcyB6OgogICAgICAgICAgICAgICAgZm9yIGl0ZW0gaW4gei5pbmZvbGlzdCgpOgogICAgICAgICAgICAgICAgICAgIHBhdGggPSBQYXRoKGl0ZW0uZmlsZW5hbWUpCiAgICAgICAgICAgICAgICAgICAgYXNzZXJ0IG5vdCBwYXRoLmlzX2Fic29sdXRlKCkgYW5kICcuLicgbm90IGluIHBhdGgucGFydHMKICAgICAgICAgICAgICAgICAgICBhc3NlcnQgbm90IHN0YXQuU19JU0xOSyhpdGVtLmV4dGVybmFsX2F0dHIgPj4gMTYpLCAnQXJjaGl2ZSBsaW5rcyByZWplY3RlZC4nCiAgICAgICAgICAgICAgICB6LmV4dHJhY3RhbGwod29yaykKICAgICAgICBlbHNlOgogICAgICAgICAgICBzb3VyY2VfdHJhaW4gPSByZWNvdmVyeV9pbnB1dCAvICd0cmFpbicKICAgICAgICAgICAgYXNzZXJ0IChzb3VyY2VfdHJhaW4gLyAnY2hlY2twb2ludHMvbGF0ZXN0Lmpzb24nKS5pc19maWxlKCksICdDb21wbGV0ZSByZWNvdmVyeSBpbnB1dCBpcyByZXF1aXJlZC4nCiAgICAgICAgICAgIHNodXRpbC5jb3B5dHJlZShzb3VyY2VfdHJhaW4sIHRyYWluLCBjb3B5X2Z1bmN0aW9uPWJ1ZmZlcmVkX2NvcHkpCiAgICBwcmludCgnVmVyaWZ5aW5nIGVwb2NoLTMwIGNoZWNrcG9pbnQsIE1MZmxvdyBoaXN0b3J5IGFuZCBvcHRpbWl6ZXIgcGF5bG9hZC4nLCBmbHVzaD1UcnVlKQogICAgcmVjb3JkID0gdmVyaWZ5X3JlY292ZXJ5KHRyYWluLCByb290IC8gJ21hbmlmZXN0Lmpzb24nKQogICAgbGFzdCA9IHRyYWluIC8gJ2NoZWNrcG9pbnRzJyAvIGYnZXBvY2gte3JlY29yZFsiZXBvY2giXTowNGR9JyAvICdsYXN0LnB0JwogICAgcGF5bG9hZCA9IHRvcmNoLmxvYWQobGFzdCwgbWFwX2xvY2F0aW9uPSdjcHUnLCB3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICBhc3NlcnQgcGF5bG9hZC5nZXQoJ29wdGltaXplcicpIGlzIG5vdCBOb25lLCAnT3B0aW1pemVyIHN0YXRlIG1pc3Npbmc7IGNhbm5vdCByZXN1bWUgc2FtZSB0cmFpbmluZy4nCiAgICBhc3NlcnQgcGF5bG9hZFsnZXBvY2gnXSArIDEgPT0gcmVjb3JkWydlcG9jaCddLCAnQ2hlY2twb2ludCBlcG9jaCBkaWZmZXJzIGZyb20gc2F2ZWQgbWV0YWRhdGEuJwogICAgYXJncyA9IHBheWxvYWRbJ3RyYWluX2FyZ3MnXQogICAgYXNzZXJ0IChhcmdzWydpbWdzeiddLCBhcmdzWydiYXRjaCddLCBhcmdzWydlcG9jaHMnXSwgYXJnc1sncGF0aWVuY2UnXSkgPT0gKDk2MCwgNCwgMTAwLCAyMCkKICAgIHJlY29yZC51cGRhdGUoc3RhdHVzPSdwcmVmbGlnaHRfcGFzc2VkX3RyYWluaW5nX25vdF9zdGFydGVkJywgdmVyc2lvbnM9YWN0dWFsLAogICAgICAgICAgICAgICAgICBvcHRpbWl6ZXJfc3RhdGVfdmVyaWZpZWQ9VHJ1ZSwgY2hlY2twb2ludF9pbnRlcm5hbF9lcG9jaD1wYXlsb2FkWydlcG9jaCddLAogICAgICAgICAgICAgICAgICBkYXRhc2V0X2NvdW50cz17J3RyYWluJzogNzAwMCwgJ3ZhbGlkYXRpb24nOiAxNTAwLCAndGVzdCc6IDB9LAogICAgICAgICAgICAgICAgICBncHU9dG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCksCiAgICAgICAgICAgICAgICAgIGR1cmFibGVfYmFja3VwX3ZlcmlmaWVkPUZhbHNlLAogICAgICAgICAgICAgICAgICBuZXh0X2dhdGU9J1ZlcmlmeSBkdXJhYmxlIHByaXZhdGUgYmFja3VwcyBhbmQgc2FtZS1ydW4gcmVzdG9yYXRpb24gYmVmb3JlIGFueSB0cmFpbmluZy4nKQogICAgZGVsIHBheWxvYWQKICAgICh3b3JrIC8gJ3ByZWZsaWdodC5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlY29yZCwgaW5kZW50PTIpKQogICAgcHJpbnQoanNvbi5kdW1wcyhyZWNvcmQsIGluZGVudD0yKSwgZmx1c2g9VHJ1ZSkKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgbWFpbigpCg==", "kaggle_backup.py": "IiIiUHJpdmF0ZSByZW1vdGUgY2hlY2twb2ludCB2ZXJzaW9uczsgbG9jYWwgS2FnZ2xlIGZpbGVzIGFsb25lIGFyZSBub3QgZHVyYWJsZS4iIiIKaW1wb3J0IGNvbnRleHRsaWIKaW1wb3J0IGlvCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc2h1dGlsCmltcG9ydCB0YXJmaWxlCmltcG9ydCB0aW1lCmltcG9ydCB1dWlkCmltcG9ydCB6aXBmaWxlCgpmcm9tIHJ1bm5lciBpbXBvcnQgc2hhLCB2ZXJpZnlfbGF0ZXN0CmZyb20gdmVyaWZ5X21pZ3JhdGlvbiBpbXBvcnQgUlVOX0lECgoKY2xhc3MgS2FnZ2xlQmFja3VwOgogICAgc2x1ZyA9ICdhYXl1c2hibWxvcHMyNi9tbG9wcy1lbC15b2xvMTFtLWNoZWNrcG9pbnRzLXByaXZhdGUnCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGFwaSk6CiAgICAgICAgc2VsZi5hcGkgPSBhcGkKICAgICAgICBzZWxmLnZlcmlmaWVkX2Vwb2NoID0gTm9uZQogICAgICAgIHNlbGYuc3RhZ2UgPSBQYXRoKCcva2FnZ2xlL3RlbXAvbWxvcHNfYmFja3VwX3VwbG9hZCcpCiAgICAgICAgc2VsZi5zdGFnZS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgZGVmIGlzX3ByaXZhdGUoc2VsZik6CiAgICAgICAgaWYgaGFzYXR0cihzZWxmLmFwaSwgJ2RhdGFzZXRfdmlldycpOgogICAgICAgICAgICB2aWV3ID0gc2VsZi5hcGkuZGF0YXNldF92aWV3KHNlbGYuc2x1ZykKICAgICAgICAgICAgZGF0YSA9IHZpZXcudG9fZGljdCgpIGlmIGhhc2F0dHIodmlldywgJ3RvX2RpY3QnKSBlbHNlIHZhcnModmlldykKICAgICAgICAgICAgcHJpdmF0ZSA9IGRhdGEuZ2V0KCdpc19wcml2YXRlJywgZGF0YS5nZXQoJ2lzUHJpdmF0ZScpKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgICMgVGhlIGF1dGhlbnRpY2F0ZWQgb3duZWQtZGF0YXNldCBsaXN0aW5nIGV4cG9zZXMgdGhlIHByaXZhY3kgYml0CiAgICAgICAgICAgICMgZXZlbiB3aGlsZSBhIG5ld2x5IGNyZWF0ZWQgZGF0YXNldCdzIGRldGFpbCBlbmRwb2ludCBpcyBwZW5kaW5nLgogICAgICAgICAgICBtYXRjaGVzID0gW2l0ZW0gZm9yIGl0ZW0gaW4gKHNlbGYuYXBpLmRhdGFzZXRfbGlzdChtaW5lPVRydWUpIG9yIFtdKQogICAgICAgICAgICAgICAgICAgICAgIGlmIGdldGF0dHIoaXRlbSwgJ3JlZicsIE5vbmUpID09IHNlbGYuc2x1Z10KICAgICAgICAgICAgcHJpdmF0ZSA9IGdldGF0dHIobWF0Y2hlc1swXSwgJ2lzX3ByaXZhdGUnLCBOb25lKSBpZiBsZW4obWF0Y2hlcykgPT0gMSBlbHNlIE5vbmUKICAgICAgICBpZiBwcml2YXRlIGlzIG5vdCBUcnVlOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1JlbW90ZSBjaGVja3BvaW50IGRhdGFzZXQgaXMgbm90IGNvbmZpcm1lZCBwcml2YXRlJykKCiAgICBkZWYgZG93bmxvYWRfZmlsZShzZWxmLCBuYW1lLCBkZXN0aW5hdGlvbik6CiAgICAgICAgZGVzdGluYXRpb24gPSBQYXRoKGRlc3RpbmF0aW9uKQogICAgICAgIHRhcmdldCA9IGRlc3RpbmF0aW9uIC8gbmFtZQogICAgICAgICMgS2FnZ2xlIDIuMC4yIHdyYXBzIGxhcmdlIGRvd25sb2FkcyBpbiBaSVAgYW5kIGhhcyBubyB1bnppcCBhcmd1bWVudC4KICAgICAgICB0YXJnZXQudW5saW5rKG1pc3Npbmdfb2s9VHJ1ZSkKICAgICAgICAoZGVzdGluYXRpb24gLyAobmFtZSArICcuemlwJykpLnVubGluayhtaXNzaW5nX29rPVRydWUpCiAgICAgICAgd2l0aCBjb250ZXh0bGliLnJlZGlyZWN0X3N0ZG91dChpby5TdHJpbmdJTygpKToKICAgICAgICAgICAgc2VsZi5hcGkuZGF0YXNldF9kb3dubG9hZF9maWxlKHNlbGYuc2x1ZywgbmFtZSwgc3RyKGRlc3RpbmF0aW9uKSwgZm9yY2U9VHJ1ZSwgcXVpZXQ9VHJ1ZSkKICAgICAgICBpZiBub3QgdGFyZ2V0LmlzX2ZpbGUoKToKICAgICAgICAgICAgd3JhcHBlZCA9IGRlc3RpbmF0aW9uIC8gKG5hbWUgKyAnLnppcCcpCiAgICAgICAgICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKHdyYXBwZWQpIGFzIHo6CiAgICAgICAgICAgICAgICBtZW1iZXJzID0gei5pbmZvbGlzdCgpCiAgICAgICAgICAgICAgICBpZiBsZW4obWVtYmVycykgIT0gMSBvciBQYXRoKG1lbWJlcnNbMF0uZmlsZW5hbWUpLm5hbWUgIT0gbmFtZSBvciBtZW1iZXJzWzBdLmlzX2RpcigpOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1VuZXhwZWN0ZWQgd3JhcHBlZCBkb3dubG9hZCBtZW1iZXJzJykKICAgICAgICAgICAgICAgIHdpdGggei5vcGVuKG1lbWJlcnNbMF0pIGFzIHNyYywgdGFyZ2V0Lm9wZW4oJ3diJykgYXMgZHN0OgogICAgICAgICAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihzcmMsIGRzdCwgMTAyNCAqIDEwMjQpCiAgICAgICAgICAgIHdyYXBwZWQudW5saW5rKCkKICAgICAgICByZXR1cm4gdGFyZ2V0CgogICAgZGVmIHB1Ymxpc2goc2VsZiwgcGVyc2lzdGVudCwgY3JlYXRlPUZhbHNlKToKICAgICAgICBwZXJzaXN0ZW50ID0gUGF0aChwZXJzaXN0ZW50KQogICAgICAgIF8sIHN0YXRlID0gdmVyaWZ5X2xhdGVzdChwZXJzaXN0ZW50KQogICAgICAgIGlmIHN0YXRlWydydW5faWQnXSAhPSBSVU5fSUQgb3Igc3RhdGVbJ21vZGUnXSAhPSAndHJhaW4nOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdSZWZ1c2luZyBiYWNrdXAgZm9yIGFub3RoZXIgZXhwZXJpbWVudCcpCiAgICAgICAgaWYgbm90IGNyZWF0ZToKICAgICAgICAgICAgc2VsZi5pc19wcml2YXRlKCkKICAgICAgICAjIE9ubHkgdGhlIGxhdGVzdCBjb21wbGV0ZSBnZW5lcmF0aW9uIGdvZXMgaW4gZWFjaCByZW1vdGUgdmVyc2lvbi4KICAgICAgICAjIEVhcmxpZXIgcmVtb3RlIHZlcnNpb25zIHByZXNlcnZlIGVhcmxpZXIgZ2VuZXJhdGlvbnM7IG5vIHJlbW90ZSBkZWxldGlvbiBvY2N1cnMuCiAgICAgICAgYXJjaGl2ZSA9IHNlbGYuc3RhZ2UgLyAncmVjb3ZlcnkuYmluJwogICAgICAgIGdlbmVyYXRpb24gPSAnY2hlY2twb2ludHMvJyArIHN0YXRlWydnZW5lcmF0aW9uJ10KICAgICAgICB3aXRoIHRhcmZpbGUub3BlbihhcmNoaXZlLCAndzonKSBhcyB0YXI6CiAgICAgICAgICAgIGZvciByZWxhdGl2ZSBpbiAoJ2NoZWNrcG9pbnRzL2xhdGVzdC5qc29uJywgZ2VuZXJhdGlvbiwgJ21sZmxvdy5kYicsICdtbGFydGlmYWN0cycsICd0cmFpbmluZycpOgogICAgICAgICAgICAgICAgcGF0aCA9IHBlcnNpc3RlbnQgLyByZWxhdGl2ZQogICAgICAgICAgICAgICAgaWYgbm90IHBhdGguZXhpc3RzKCk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZidNaXNzaW5nIHJlY292ZXJ5IGFydGlmYWN0OiB7cmVsYXRpdmV9JykKICAgICAgICAgICAgICAgIHRhci5hZGQocGF0aCwgYXJjbmFtZT0ndHJhaW4vJyArIHJlbGF0aXZlLCByZWN1cnNpdmU9VHJ1ZSkKICAgICAgICAgICAgZm9yIG9wdGlvbmFsIGluICgndHJhaW5fcmVwb3J0Lmpzb24nLCAneW9sbzExbS13YXN0ZS1iZXN0LnB0Jyk6CiAgICAgICAgICAgICAgICBpZiAocGVyc2lzdGVudCAvIG9wdGlvbmFsKS5pc19maWxlKCk6CiAgICAgICAgICAgICAgICAgICAgdGFyLmFkZChwZXJzaXN0ZW50IC8gb3B0aW9uYWwsIGFyY25hbWU9J3RyYWluLycgKyBvcHRpb25hbCkKICAgICAgICByZWNlaXB0ID0geydydW5faWQnOiBSVU5fSUQsICdlcG9jaCc6IHN0YXRlWydlcG9jaCddLCAncHJvdmlkZXInOiAna2FnZ2xlJywKICAgICAgICAgICAgICAgICAgICdyZWNvdmVyeV9zaGEyNTYnOiBzaGEoYXJjaGl2ZSksICdyZWNvdmVyeV9ieXRlcyc6IGFyY2hpdmUuc3RhdCgpLnN0X3NpemUsCiAgICAgICAgICAgICAgICAgICAnY2hlY2twb2ludF9maWxlcyc6IHN0YXRlWydmaWxlcyddLCAnbm9uY2UnOiB1dWlkLnV1aWQ0KCkuaGV4fQogICAgICAgIChzZWxmLnN0YWdlIC8gJ3JlY2VpcHQuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZWNlaXB0LCBpbmRlbnQ9MikpCiAgICAgICAgKHNlbGYuc3RhZ2UgLyAnZGF0YXNldC1tZXRhZGF0YS5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKHsKICAgICAgICAgICAgJ2lkJzogc2VsZi5zbHVnLCAndGl0bGUnOiAnTUxPUFMgRUwgWU9MTzExbSBjaGVja3BvaW50cyBwcml2YXRlJywKICAgICAgICAgICAgJ2xpY2Vuc2VzJzogW3snbmFtZSc6ICdvdGhlcid9XX0sIGluZGVudD0yKSkKICAgICAgICBpZiBjcmVhdGU6CiAgICAgICAgICAgIHJlc3BvbnNlID0gc2VsZi5hcGkuZGF0YXNldF9jcmVhdGVfbmV3KHN0cihzZWxmLnN0YWdlKSwgcHVibGljPUZhbHNlLCBxdWlldD1UcnVlLCBjb252ZXJ0X3RvX2Nzdj1GYWxzZSkKICAgICAgICBlbHNlOgogICAgICAgICAgICByZXNwb25zZSA9IHNlbGYuYXBpLmRhdGFzZXRfY3JlYXRlX3ZlcnNpb24oc3RyKHNlbGYuc3RhZ2UpLCBmJ1NhbWUgcnVuIGVwb2NoIHtzdGF0ZVsiZXBvY2giXX0nLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcXVpZXQ9VHJ1ZSwgY29udmVydF90b19jc3Y9RmFsc2UsIGRlbGV0ZV9vbGRfdmVyc2lvbnM9RmFsc2UpCiAgICAgICAgaWYgZ2V0YXR0cihyZXNwb25zZSwgJ2Vycm9yJywgTm9uZSk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignS2FnZ2xlIHJlamVjdGVkIHRoZSBjaGVja3BvaW50IHZlcnNpb247IGluc3BlY3QgcmVtb3RlIGRhdGFzZXQgYmVmb3JlIHJldHJ5aW5nLicpCiAgICAgICAgZG93bmxvYWQgPSBQYXRoKCcva2FnZ2xlL3RlbXAvbWxvcHNfYmFja3VwX3ZlcmlmeScpCiAgICAgICAgZG93bmxvYWQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIGRlYWRsaW5lID0gdGltZS5tb25vdG9uaWMoKSArIDkwMAogICAgICAgIHdoaWxlIHRpbWUubW9ub3RvbmljKCkgPCBkZWFkbGluZToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgc2VsZi5pc19wcml2YXRlKCkKICAgICAgICAgICAgICAgIGlmIHNlbGYuYXBpLmRhdGFzZXRfc3RhdHVzKHNlbGYuc2x1ZykubG93ZXIoKSAhPSAncmVhZHknOgogICAgICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAoMTApCiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgICMgU3VwcHJlc3MgU0RLIFVSTC9saWNlbnNlIGNoYXR0ZXI7IG5ldmVyIGxvZyBjcmVkZW50aWFscyBvciByYXcgZXJyb3IgcmVzcG9uc2VzLgogICAgICAgICAgICAgICAgc2VsZi5kb3dubG9hZF9maWxlKCdyZWNlaXB0Lmpzb24nLCBkb3dubG9hZCkKICAgICAgICAgICAgICAgIGFjdHVhbCA9IGpzb24ubG9hZHMoKGRvd25sb2FkIC8gJ3JlY2VpcHQuanNvbicpLnJlYWRfdGV4dCgpKQogICAgICAgICAgICAgICAgaWYgYWN0dWFsICE9IHJlY2VpcHQ6CiAgICAgICAgICAgICAgICAgICAgdGltZS5zbGVlcCgxMCkKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgc2VsZi5kb3dubG9hZF9maWxlKCdyZWNvdmVyeS5iaW4nLCBkb3dubG9hZCkKICAgICAgICAgICAgICAgIHJlc3RvcmVkID0gZG93bmxvYWQgLyAncmVjb3ZlcnkuYmluJwogICAgICAgICAgICAgICAgaWYgcmVzdG9yZWQuc3RhdCgpLnN0X3NpemUgIT0gcmVjZWlwdFsncmVjb3ZlcnlfYnl0ZXMnXSBvciBzaGEocmVzdG9yZWQpICE9IHJlY2VpcHRbJ3JlY292ZXJ5X3NoYTI1NiddOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignUmVtb3RlIHJlY292ZXJ5IGFyY2hpdmUgY2hlY2tzdW0gbWlzbWF0Y2gnKQogICAgICAgICAgICAgICAgc2VsZi52ZXJpZmllZF9lcG9jaCA9IHN0YXRlWydlcG9jaCddCiAgICAgICAgICAgICAgICAocGVyc2lzdGVudCAvICdrYWdnbGUtcmVtb3RlLXJlY2VpcHQuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZWNlaXB0LCBpbmRlbnQ9MikpCiAgICAgICAgICAgICAgICAjIFJlbW92ZSBkaXNwb3NhYmxlIHZlcmlmaWNhdGlvbiBkb3dubG9hZHMgYWZ0ZXIgYSBzdWNjZXNzZnVsIGhhc2ggY2hlY2suCiAgICAgICAgICAgICAgICBzaHV0aWwucm10cmVlKGRvd25sb2FkKQogICAgICAgICAgICAgICAgcHJpbnQoZidQUklWQVRFIFJFTU9URSBCQUNLVVAgVkVSSUZJRUQ6IGVwb2NoIHtzZWxmLnZlcmlmaWVkX2Vwb2NofScsIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICByZXR1cm4gcmVjZWlwdAogICAgICAgICAgICBleGNlcHQgKEZpbGVOb3RGb3VuZEVycm9yLCBqc29uLkpTT05EZWNvZGVFcnJvcik6CiAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKDEwKQogICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcignUmVtb3RlIGNoZWNrcG9pbnQgdmVyc2lvbiB3YXMgbm90IHZlcmlmaWVkOyBzdG9wIHRyYWluaW5nIGFuZCBpbnNwZWN0IHJlY292ZXJ5LicpCgogICAgZGVmIHZhbGlkYXRlKHNlbGYsIHByb2plY3QsIG1vZGUsIHJlc3VtZSk6CiAgICAgICAgcHJvamVjdCA9IFBhdGgocHJvamVjdCkucmVzb2x2ZSgpCiAgICAgICAgaWYgc3RyKHByb2plY3QpICE9ICcva2FnZ2xlL3dvcmtpbmcvTUxPUFNfRUxfY2xvdWQnIG9yIG1vZGUgIT0gJ3RyYWluJyBvciBub3QgcmVzdW1lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdLYWdnbGUgYWRhcHRlciBwZXJtaXRzIG9ubHkgdGhlIGV4aXN0aW5nIHJlc3VtZWQgZnVsbCBleHBlcmltZW50JykKICAgICAgICBfLCBzdGF0ZSA9IHZlcmlmeV9sYXRlc3QocHJvamVjdCAvIG1vZGUpCiAgICAgICAgaWYgc2VsZi52ZXJpZmllZF9lcG9jaCAhPSBzdGF0ZVsnZXBvY2gnXToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdDdXJyZW50IHJlY292ZXJ5IGdlbmVyYXRpb24gaGFzIG5vdCBwYXNzZWQgcmVtb3RlIHJvdW5kLXRyaXAgdmVyaWZpY2F0aW9uJykKICAgICAgICBzZWxmLmlzX3ByaXZhdGUoKQoKICAgIGRlZiByZXN0b3JlKHNlbGYsIGRlc3RpbmF0aW9uKToKICAgICAgICAiIiJGZXRjaCBhIGNvbXBsZXRlIHJlbW90ZSBnZW5lcmF0aW9uIGludG8gYSBuZXcsIGVtcHR5IHJlY292ZXJ5IGRpcmVjdG9yeS4iIiIKICAgICAgICBkZXN0aW5hdGlvbiA9IFBhdGgoZGVzdGluYXRpb24pCiAgICAgICAgaWYgZGVzdGluYXRpb24uZXhpc3RzKCk6CiAgICAgICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcignUmVjb3ZlcnkgZGVzdGluYXRpb24gbXVzdCBiZSBmcmVzaDsgcHJlc2VydmUgZXhpc3RpbmcgYXR0ZW1wdHMnKQogICAgICAgIHNlbGYuaXNfcHJpdmF0ZSgpCiAgICAgICAgZG93bmxvYWQgPSBQYXRoKCcva2FnZ2xlL3RlbXAvbWxvcHNfcmVtb3RlX3Jlc3RvcmUnKQogICAgICAgIGRvd25sb2FkLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBmb3IgbmFtZSBpbiAoJ3JlY2VpcHQuanNvbicsICdyZWNvdmVyeS5iaW4nKToKICAgICAgICAgICAgc2VsZi5kb3dubG9hZF9maWxlKG5hbWUsIGRvd25sb2FkKQogICAgICAgIHJlY2VpcHQgPSBqc29uLmxvYWRzKChkb3dubG9hZCAvICdyZWNlaXB0Lmpzb24nKS5yZWFkX3RleHQoKSkKICAgICAgICBhcmNoaXZlID0gZG93bmxvYWQgLyAncmVjb3ZlcnkuYmluJwogICAgICAgIGlmIHJlY2VpcHRbJ3J1bl9pZCddICE9IFJVTl9JRCBvciBzaGEoYXJjaGl2ZSkgIT0gcmVjZWlwdFsncmVjb3Zlcnlfc2hhMjU2J106CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1JlbW90ZSByZWNvdmVyeSBpZGVudGl0eS9oYXNoIG1pc21hdGNoJykKICAgICAgICB3aXRoIHRhcmZpbGUub3BlbihhcmNoaXZlLCAncjonKSBhcyB0YXI6CiAgICAgICAgICAgIGZyb20gcnVubmVyIGltcG9ydCBzYWZlX25hbWUKICAgICAgICAgICAgZm9yIG1lbWJlciBpbiB0YXIuZ2V0bWVtYmVycygpOgogICAgICAgICAgICAgICAgcGF0aCA9IHNhZmVfbmFtZShtZW1iZXIubmFtZSkKICAgICAgICAgICAgICAgIGlmIHBhdGgucGFydHNbMF0gIT0gJ3RyYWluJyBvciBub3QgKG1lbWJlci5pc2ZpbGUoKSBvciBtZW1iZXIuaXNkaXIoKSk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVW5zYWZlIHJlY292ZXJ5IGFyY2hpdmUgbWVtYmVyJykKICAgICAgICAgICAgZGVzdGluYXRpb24ubWtkaXIocGFyZW50cz1UcnVlKQogICAgICAgICAgICB0YXIuZXh0cmFjdGFsbChkZXN0aW5hdGlvbiwgZmlsdGVyPSdkYXRhJykKICAgICAgICB2ZXJpZnlfbGF0ZXN0KGRlc3RpbmF0aW9uIC8gJ3RyYWluJykKICAgICAgICBzaHV0aWwucm10cmVlKGRvd25sb2FkKQogICAgICAgIHJldHVybiByZWNlaXB0Cg==", "start_kaggle_training.py": "IiIiUmVzdW1lIHRoZSBzYW1lIGV4cGVyaW1lbnQgb25seSBhZnRlciBwcml2YXRlIHJlbW90ZSBiYWNrdXAgcm91bmQtdHJpcCBzdWNjZWVkcy4iIiIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YQppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKaW1wb3J0IHRvcmNoCmltcG9ydCBydW5uZXIKZnJvbSBrYWdnbGVfYmFja3VwIGltcG9ydCBLYWdnbGVCYWNrdXAKZnJvbSB2ZXJpZnlfbWlncmF0aW9uIGltcG9ydCBSVU5fSUQsIEVYUEVDVEVEX1ZFUlNJT05TLCB2ZXJpZnlfdmVyc2lvbnMsIHZlcmlmeV9yZWNvdmVyeQoKCmRlZiBsaW5rX2V4aXN0aW5nX3BhdGgoYWxpYXMsIHRhcmdldCk6CiAgICBhbGlhcywgdGFyZ2V0ID0gUGF0aChhbGlhcyksIFBhdGgodGFyZ2V0KQogICAgaWYgYWxpYXMuZXhpc3RzKCkgb3IgYWxpYXMuaXNfc3ltbGluaygpOgogICAgICAgIGlmIGFsaWFzLnJlc29sdmUoKSAhPSB0YXJnZXQucmVzb2x2ZSgpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZidFeGlzdGluZyBwYXRoIGNvbmZsaWN0cyB3aXRoIG9yaWdpbmFsIE1MZmxvdy9jaGVja3BvaW50IHBhdGg6IHthbGlhc30nKQogICAgICAgIHJldHVybgogICAgYWxpYXMucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGFsaWFzLnN5bWxpbmtfdG8odGFyZ2V0LCB0YXJnZXRfaXNfZGlyZWN0b3J5PVRydWUpCgoKZGVmIG1haW4oaW5pdGlhbGl6ZV9iYWNrdXA9RmFsc2UpOgogICAgYXNzZXJ0IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCksICdGcmVlIENVREEgR1BVIHJlcXVpcmVkOyBubyBDUFUgZmFsbGJhY2suJwogICAgdmVyaWZ5X3ZlcnNpb25zKHtuYW1lOiBpbXBvcnRsaWIubWV0YWRhdGEudmVyc2lvbihuYW1lKSBmb3IgbmFtZSBpbiBFWFBFQ1RFRF9WRVJTSU9OU30pCiAgICByb290ID0gUGF0aCgnL2thZ2dsZS90ZW1wL21sb3BzX3RyYWludmFsX2J1bmRsZScpCiAgICBtYW5pZmVzdCA9IHJ1bm5lci52ZXJpZnlfYnVuZGxlKHJvb3QpCiAgICBhc3NlcnQgbWFuaWZlc3RbJ3Njb3BlJ10gPT0gJ3RyYWludmFsJwogICAgcmVwb3J0ID0ganNvbi5sb2FkcyhQYXRoKCcva2FnZ2xlL3dvcmtpbmcvbWxvcHNfbWlncmF0aW9uL3ByZWZsaWdodC5qc29uJykucmVhZF90ZXh0KCkpCiAgICBhc3NlcnQgcmVwb3J0WydvcHRpbWl6ZXJfc3RhdGVfdmVyaWZpZWQnXSBhbmQgcmVwb3J0WydydW5faWQnXSA9PSBSVU5fSUQKICAgIGZyb20ga2FnZ2xlX3NlY3JldHMgaW1wb3J0IFVzZXJTZWNyZXRzQ2xpZW50CiAgICBvcy5lbnZpcm9uWydLQUdHTEVfQVBJX1RPS0VOJ10gPSBVc2VyU2VjcmV0c0NsaWVudCgpLmdldF9zZWNyZXQoJ01MT1BTX0VMX0tBR0dMRV9BUElfVE9LRU4nKQogICAgZnJvbSBrYWdnbGUuYXBpLmthZ2dsZV9hcGlfZXh0ZW5kZWQgaW1wb3J0IEthZ2dsZUFwaQogICAgYXBpID0gS2FnZ2xlQXBpKCkKICAgIGFwaS5hdXRoZW50aWNhdGUoKQogICAgYmFja2VuZCA9IEthZ2dsZUJhY2t1cChhcGkpCiAgICBwcm9qZWN0ID0gUGF0aCgnL2thZ2dsZS93b3JraW5nL01MT1BTX0VMX2Nsb3VkJykKICAgIHRyYWluID0gcHJvamVjdCAvICd0cmFpbicKICAgIGlmIGluaXRpYWxpemVfYmFja3VwOgogICAgICAgIGlmIHRyYWluLmV4aXN0cygpOgogICAgICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoJ0luaXRpYWwgbWlncmF0aW9uIGF0dGVtcHQgYWxyZWFkeSBleGlzdHM7IGluc3BlY3QgYmVmb3JlIHJldHJ5aW5nLicpCiAgICAgICAgcHJvamVjdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgc2h1dGlsLmNvcHl0cmVlKCcva2FnZ2xlL3dvcmtpbmcvbWxvcHNfbWlncmF0aW9uL3RyYWluJywgdHJhaW4pCiAgICBlbHNlOgogICAgICAgICMgUmVzdG9yZSB0aGUgYXV0aG9yaXRhdGl2ZSByZW1vdGUgdmVyc2lvbiwgbmV2ZXIgc2lsZW50bHkgcmV1c2UgYW4gb2xkZXIgaW5wdXQgc25hcHNob3QuCiAgICAgICAgaW1wb3J0ZWQgPSBQYXRoKCcva2FnZ2xlL3RlbXAnKSAvICgnbWxvcHMtcmVzdW1lLScgKyBzdHIodGltZS50aW1lX25zKCkpKQogICAgICAgIGJhY2tlbmQucmVzdG9yZShpbXBvcnRlZCkKICAgICAgICBpZiB0cmFpbi5leGlzdHMoKToKICAgICAgICAgICAgbG9jYWxfbGFzdCwgbG9jYWxfc3RhdGUgPSBydW5uZXIudmVyaWZ5X2xhdGVzdCh0cmFpbikKICAgICAgICAgICAgcmVtb3RlX2xhc3QsIHJlbW90ZV9zdGF0ZSA9IHJ1bm5lci52ZXJpZnlfbGF0ZXN0KGltcG9ydGVkIC8gJ3RyYWluJykKICAgICAgICAgICAgaWYgbG9jYWxfc3RhdGVbJ2Vwb2NoJ10gPiByZW1vdGVfc3RhdGVbJ2Vwb2NoJ106CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0xvY2FsIHByb2dyZXNzIGlzIG5ld2VyIHRoYW4gcmVtb3RlOiBwcmVzZXJ2ZSBhbmQgaW5zcGVjdCBiZWZvcmUgcmVzdW1pbmcuJykKICAgICAgICAgICAgdHJhaW4ucmVuYW1lKHByb2plY3QgLyAoJ3RyYWluLXByZXNlcnZlZC0nICsgc3RyKHRpbWUudGltZV9ucygpKSkpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcHJvamVjdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgc2h1dGlsLmNvcHl0cmVlKGltcG9ydGVkIC8gJ3RyYWluJywgdHJhaW4pCiAgICBzdGF0ZSA9IHZlcmlmeV9yZWNvdmVyeSh0cmFpbiwgcm9vdCAvICdtYW5pZmVzdC5qc29uJykKICAgIGlmICh0cmFpbiAvICd0cmFpbl9yZXBvcnQuanNvbicpLmV4aXN0cygpOgogICAgICAgIGlmIGpzb24ubG9hZHMoKHRyYWluIC8gJ3RyYWluX3JlcG9ydC5qc29uJykucmVhZF90ZXh0KCkpLmdldCgnc3RhdHVzJykgPT0gJ2NvbXBsZXRlZCc6CiAgICAgICAgICAgIHByaW50KCdFeGlzdGluZyBleHBlcmltZW50IGNvbXBsZXRlZDsgbm8gdHJhaW5pbmcgc3RhcnRlZC4nLCBmbHVzaD1UcnVlKQogICAgICAgICAgICByZXR1cm4KICAgIGxhc3QsIHBvaW50ZXIgPSBydW5uZXIudmVyaWZ5X2xhdGVzdCh0cmFpbikKICAgIHBheWxvYWQgPSB0b3JjaC5sb2FkKGxhc3QsIG1hcF9sb2NhdGlvbj0nY3B1Jywgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgYXNzZXJ0IHBheWxvYWQuZ2V0KCdvcHRpbWl6ZXInKSBpcyBub3QgTm9uZSBhbmQgcGF5bG9hZFsnZXBvY2gnXSArIDEgPT0gc3RhdGVbJ2Vwb2NoJ10KICAgIGFyZ3MgPSBwYXlsb2FkWyd0cmFpbl9hcmdzJ10KICAgIGFzc2VydCAoYXJnc1snaW1nc3onXSwgYXJnc1snYmF0Y2gnXSwgYXJnc1snZXBvY2hzJ10sIGFyZ3NbJ3BhdGllbmNlJ10pID09ICg5NjAsIDQsIDEwMCwgMjApCiAgICBkZWwgcGF5bG9hZAogICAgd29yayA9IFBhdGgoJy9rYWdnbGUvd29ya2luZy9tbG9wc193b3JrJykKICAgIHdvcmsubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgbGlua19leGlzdGluZ19wYXRoKCcvY29udGVudC9tbG9wc193b3JrJywgd29yaykKICAgIGxpbmtfZXhpc3RpbmdfcGF0aCgnL2NvbnRlbnQvbWxvcHNfdHJhaW52YWxfYnVuZGxlJywgcm9vdCkKICAgIHNlc3Npb24gPSBwcm9qZWN0IC8gJ3Nlc3Npb25zJyAvIHN0cih0aW1lLnRpbWVfbnMoKSkKICAgIHNlc3Npb24ubWtkaXIocGFyZW50cz1UcnVlKQogICAgc2Vzc2lvbl9yZWNvcmQgPSB7J3J1bl9pZCc6IFJVTl9JRCwgJ3Byb3ZpZGVyJzogJ2thZ2dsZScsICdyZXN1bWVfZXBvY2gnOiBzdGF0ZVsnZXBvY2gnXSwKICAgICAgICAgICAgICAgICAgICAgICdjb2xhYl9vbl9ob2xkJzogVHJ1ZSwgJ3NldHRpbmdzJzogeydpbWdzeic6IDk2MCwgJ2JhdGNoJzogNCwgJ2Vwb2Nocyc6IDEwMCwgJ3BhdGllbmNlJzogMjB9LAogICAgICAgICAgICAgICAgICAgICAgJ3ZlcnNpb25zJzoge25hbWU6IGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKG5hbWUpIGZvciBuYW1lIGluIEVYUEVDVEVEX1ZFUlNJT05TfSwKICAgICAgICAgICAgICAgICAgICAgICdydW5uZXJfc2hhMjU2JzogcnVubmVyLnNoYShQYXRoKHJ1bm5lci5fX2ZpbGVfXykpLCAnYmFja3VwX3NoYTI1Nic6IHJ1bm5lci5zaGEoUGF0aChzeXMubW9kdWxlc1sna2FnZ2xlX2JhY2t1cCddLl9fZmlsZV9fKSksCiAgICAgICAgICAgICAgICAgICAgICAnZ3B1JzogdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCl9CiAgICAoc2Vzc2lvbiAvICdzZXNzaW9uLmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoc2Vzc2lvbl9yZWNvcmQsIGluZGVudD0yKSkKICAgIChzZXNzaW9uIC8gJ3JlcXVpcmVtZW50cy1yZXNvbHZlZC50eHQnKS53cml0ZV90ZXh0KHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KFtzeXMuZXhlY3V0YWJsZSwgJy1tJywgJ3BpcCcsICdmcmVlemUnXSwgdGV4dD1UcnVlKSkKICAgIHByb3ZlbmFuY2UgPSB0cmFpbiAvICd0cmFpbmluZycgLyAoJ2thZ2dsZS1zZXNzaW9uLScgKyBzZXNzaW9uLm5hbWUpCiAgICBzaHV0aWwuY29weXRyZWUoc2Vzc2lvbiwgcHJvdmVuYW5jZSkKICAgIGZvciBtb2R1bGUgaW4gKFBhdGgoX19maWxlX18pLCBQYXRoKHJ1bm5lci5fX2ZpbGVfXyksIFBhdGgoc3lzLm1vZHVsZXNbJ2thZ2dsZV9iYWNrdXAnXS5fX2ZpbGVfXykpOgogICAgICAgIHNodXRpbC5jb3B5ZmlsZShtb2R1bGUsIHByb3ZlbmFuY2UgLyBtb2R1bGUubmFtZSkKICAgICMgRWFjaCBsYXRlciBvbl9tb2RlbF9zYXZlIGNhbGxiYWNrIHJlcGVhdHMgZnVsbCB1cGxvYWQvZG93bmxvYWQvaGFzaCB2ZXJpZmljYXRpb24uCiAgICBiYWNrZW5kLnB1Ymxpc2godHJhaW4sIGNyZWF0ZT1pbml0aWFsaXplX2JhY2t1cCkKICAgIHByaW50KGYnUmVzdW1pbmcgU0FNRSBNTGZsb3cgcnVuIHtSVU5fSUR9IGFmdGVyIHZlcmlmaWVkIGVwb2NoIHtzdGF0ZVsiZXBvY2giXX07IENvbGFiIGhlbGQgc3RvcHBlZC4nLCBmbHVzaD1UcnVlKQogICAgcnVubmVyLnJ1bihyb290LCBwcm9qZWN0LCBtb2RlPSd0cmFpbicsIGFwcHJvdmVfZnVsbD1UcnVlLCByZXN1bWU9VHJ1ZSwgcmVjb3ZlcnlfYmFja2VuZD1iYWNrZW5kKQoKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICBtYWluKGluaXRpYWxpemVfYmFja3VwPSctLWluaXRpYWxpemUtYmFja3VwJyBpbiBzeXMuYXJndikK"}
for name, source in embedded.items(): (code_dir/name).write_bytes(base64.b64decode(source))
sys.path.insert(0,str(code_dir))
import runpy
_ = runpy.run_path(str(code_dir/'kaggle_preflight.py'),run_name='__main__')


In [ ]:
# An existing private backup is REQUIRED; do not initialize another experiment.
import importlib, runner, kaggle_backup, start_kaggle_training
importlib.reload(runner); importlib.reload(kaggle_backup); importlib.reload(start_kaggle_training)
start_kaggle_training.main(initialize_backup=False)
